<a href="https://colab.research.google.com/github/demobut/home-tasks-and-spending-organiser/blob/main/MyHome_v4_5_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# My Home v4.5.7 — notification reliability fix

This build pins the notification dependencies, uses the official Android notification permission API, and validates notification resources before building.


## Optional — restore your existing Android signing key

If you are installing this APK **over an earlier My Home APK**, Android requires the same signing key.

If you downloaded `MyHome_signing_key.keystore` from the earlier Colab build, run the next cell and upload it.

If this is your first installation, or you are happy to uninstall the old app first, skip the next cell.

**Tip:** JSON backup/restore gives you another way to preserve your app data before uninstalling.

In [4]:
# OPTIONAL — upload an existing My Home signing key in a NEW Colab runtime
# Skip this cell for a completely fresh installation.

from google.colab import files
from pathlib import Path
import shutil

print("Choose MyHome_signing_key.keystore if you want to update an existing installed APK.")
uploaded = files.upload()

if uploaded:
    uploaded_name = next(iter(uploaded))
    target = Path.home() / ".android" / "debug.keystore"
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(uploaded_name, target)
    target.chmod(0o600)
    print("Signing key restored to:", target)
else:
    print("No key uploaded; continuing as a fresh signer.")


Choose MyHome_signing_key.keystore if you want to update an existing installed APK.


Saving MyHome_signing_key.keystore to MyHome_signing_key.keystore
Signing key restored to: /root/.android/debug.keystore


In [1]:
# @title

# CELL 1 — Install Flutter + Android SDK

import os, pathlib

!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk git unzip xz-utils zip libglu1-mesa > /dev/null

if not pathlib.Path("/content/flutter/bin/flutter").exists():
    !git clone https://github.com/flutter/flutter.git -b stable --depth 1 /content/flutter

ANDROID_SDK = "/content/android-sdk"
CMD_ZIP = "/content/commandlinetools.zip"
CMD_URL = "https://dl.google.com/android/repository/commandlinetools-linux-15859902_latest.zip"

if not pathlib.Path(f"{ANDROID_SDK}/cmdline-tools/latest/bin/sdkmanager").exists():
    !mkdir -p {ANDROID_SDK}/cmdline-tools
    !wget -q {CMD_URL} -O {CMD_ZIP}
    !rm -rf /content/cmdline-tools-unzip
    !mkdir -p /content/cmdline-tools-unzip
    !unzip -q {CMD_ZIP} -d /content/cmdline-tools-unzip
    !mkdir -p {ANDROID_SDK}/cmdline-tools/latest
    !cp -r /content/cmdline-tools-unzip/cmdline-tools/* {ANDROID_SDK}/cmdline-tools/latest/

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["ANDROID_HOME"] = ANDROID_SDK
os.environ["ANDROID_SDK_ROOT"] = ANDROID_SDK
os.environ["PATH"] = (
    "/content/flutter/bin:"
    f"{ANDROID_SDK}/cmdline-tools/latest/bin:"
    f"{ANDROID_SDK}/platform-tools:"
    + os.environ["PATH"]
)

!yes | sdkmanager --licenses > /dev/null
!sdkmanager "platform-tools" "platforms;android-36" "build-tools;36.0.0" > /dev/null

!flutter config --no-analytics
!flutter config --android-sdk {ANDROID_SDK}
!flutter --version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into '/content/flutter'...
remote: Enumerating objects: 17278, done.
remote: Counting objects: 100% (17278/17278), done.
remote: Compressing objects: 100% (13838/13838), done.
remote: Total 17278 (delta 3746), reused 9161 (delta 2539), pack-reused 0 (from 0)
Receiving objects: 100% (17278/17278), 34.20 MiB | 1.67 MiB/s, done.
Resolving deltas: 100% (3746/3746), done.
Updating files: 100% (16076/16076), done.
The 'android' binary can also be found in the cmdline-tools directory, and 'android sdk' is the replacement for 'sdkmanager'.
To learn more about the Android CLI and how to use it, see the documentation (https://d.android.com/tools/agents/android-cli)

The 'android' binary can also be found in the cmdline-tools directory, and 'android sdk' is the replacement for 'sdkmanager'.
To learn mor

In [2]:
# @title

# CELL 2 — Create project, dependencies and ALL directories

import pathlib, shutil

PROJECT = "/content/home_organiser"

if pathlib.Path(PROJECT).exists():
    shutil.rmtree(PROJECT)

!cd /content && flutter create --platforms=android --org uk.co.local home_organiser

# Core packages + Android native save/open file picker for backup/restore.
!cd {PROJECT} && flutter pub add sqflite path intl file_picker
!cd {PROJECT} && flutter pub add flutter_local_notifications:22.3.3 workmanager:0.10.10

# Remove Flutter's generated counter-app test.
!rm -rf {PROJECT}/test

# Explicitly create directories before any %%writefile cells.
!mkdir -p {PROJECT}/lib/db
!mkdir -p {PROJECT}/lib/models
!mkdir -p {PROJECT}/lib/screens
!mkdir -p {PROJECT}/lib/services
!mkdir -p {PROJECT}/android/app/src/main/res/drawable
!mkdir -p {PROJECT}/android/app/src/main/res/raw

print("Project and directories created.")


[1/10] Material Fonts
[1/10] Material Fonts                                            (2.3MB in 0.5s)
[2/10] Gradle Wrapper
[2/10] Gradle Wrapper                                            (0.1MB in 0.0s)
[3/10] Flutter SDK
  ├─ [1/5] sky_engine
  ├─ [1/5] sky_engine                                            (1.5MB in 0.2s)
  ├─ [2/5] flutter_gpu
  ├─ [2/5] flutter_gpu                                           (0.1MB in 0.0s)
  ├─ [3/5] flutter_patched_sdk
  ├─ [3/5] flutter_patched_sdk                                   (4.1MB in 0.3s)
  ├─ [4/5] flutter_patched_sdk_product
  ├─ [4/5] flutter_patched_sdk_product                           (4.1MB in 0.3s)
  └─ [5/5] linux-x64
  └─ [5/5] linux-x64                                            (29.3MB in 1.3s)
[10/10] linux-x64/font-subset
[10/10] linux-x64/font-subset                                    (2.3MB in 0.2s)
Creating project home_organiser...
Resolving dependencies in `home_organiser`...
Got dependencies in `home_organiser`.
Wrot

In [3]:
# @title
# CELL 2B — Android icon + notification build setup
from pathlib import Path
import re

PROJECT = Path('/content/home_organiser')
res_root = PROJECT / 'android/app/src/main/res'
drawable = res_root / 'drawable'
raw = res_root / 'raw'
drawable.mkdir(parents=True, exist_ok=True)
raw.mkdir(parents=True, exist_ok=True)

# Custom My Home launcher icon.
(drawable / 'my_home_icon.xml').write_text('''<vector xmlns:android="http://schemas.android.com/apk/res/android"
    android:width="108dp"
    android:height="108dp"
    android:viewportWidth="108"
    android:viewportHeight="108">
    <path android:fillColor="#315A73" android:pathData="M0,0H108V108H0Z"/>
    <path android:fillColor="#FFFFFF" android:pathData="M16,50L54,18L92,50V91H64V65H44V91H16Z"/>
    <path android:fillColor="#315A73" android:pathData="M46,49L52,55L66,40L71,45L52,65L41,54Z"/>
</vector>
''')

# Optional monochrome notification icon retained for future use. v4.5.7 uses
# the guaranteed Flutter launcher mipmap as the notification plugin default.
(drawable / 'my_home_notification.xml').write_text('''<vector xmlns:android="http://schemas.android.com/apk/res/android"
    android:width="24dp"
    android:height="24dp"
    android:viewportWidth="24"
    android:viewportHeight="24">
    <path android:fillColor="#FFFFFFFF" android:pathData="M3,11L12,3L21,11V21H14V15H10V21H3Z"/>
</vector>
''')

# flutter_local_notifications resolves icon names at runtime. Resource shrinking
# can otherwise remove resources that are only referenced by a string.
(raw / 'keep.xml').write_text('''<resources xmlns:tools="http://schemas.android.com/tools"
    tools:keep="@mipmap/ic_launcher,@drawable/my_home_icon,@drawable/my_home_notification" />
''')

manifest_path = PROJECT / 'android/app/src/main/AndroidManifest.xml'
manifest = manifest_path.read_text()
if 'android.permission.POST_NOTIFICATIONS' not in manifest:
    manifest = manifest.replace(
        '<manifest xmlns:android="http://schemas.android.com/apk/res/android">',
        '<manifest xmlns:android="http://schemas.android.com/apk/res/android">\n'
        '    <uses-permission android:name="android.permission.POST_NOTIFICATIONS" />',
    )
manifest = re.sub(r'android:label="[^"]*"', 'android:label="My Home"', manifest, count=1)
manifest = re.sub(r'android:icon="[^"]*"', 'android:icon="@drawable/my_home_icon"', manifest, count=1)
manifest_path.write_text(manifest)

# flutter_local_notifications 22.x requires core library desugaring.
gradle_path = PROJECT / 'android/app/build.gradle.kts'
gradle = gradle_path.read_text()
if 'isCoreLibraryDesugaringEnabled = true' not in gradle:
    gradle = re.sub(
        r'(compileOptions\s*\{)',
        r'\1\n        isCoreLibraryDesugaringEnabled = true',
        gradle,
        count=1,
    )
if 'coreLibraryDesugaring(' not in gradle:
    gradle += '\n\ndependencies {\n    coreLibraryDesugaring("com.android.tools:desugar_jdk_libs:2.1.4")\n}\n'
gradle_path.write_text(gradle)

# Use Flutter's normal Activity. Notification permission is requested through
# flutter_local_notifications itself, avoiding a second native MethodChannel.
main_activity = PROJECT / 'android/app/src/main/kotlin/uk/co/local/home_organiser/MainActivity.kt'
main_activity.parent.mkdir(parents=True, exist_ok=True)
main_activity.write_text('''package uk.co.local.home_organiser

import io.flutter.embedding.android.FlutterActivity

class MainActivity : FlutterActivity()
''')

print('Android icon, notification permission, resource keep rules and desugaring configured.')


Android icon, notification permission, resource keep rules and desugaring configured.


In [5]:
# @title

%%writefile /content/home_organiser/lib/models/app_models.dart
class Room {
  final int? id;
  final String name;

  const Room({this.id, required this.name});

  factory Room.fromMap(Map<String, Object?> map) =>
      Room(id: map['id'] as int?, name: map['name'] as String);
}

class Chore {
  final int? id;
  final int roomId;
  final String name;
  final int frequencyDays;
  final String? lastCompleted;
  final String nextDue;
  final String notes;

  const Chore({
    this.id,
    required this.roomId,
    required this.name,
    required this.frequencyDays,
    this.lastCompleted,
    required this.nextDue,
    this.notes = '',
  });

  factory Chore.fromMap(Map<String, Object?> map) => Chore(
        id: map['id'] as int?,
        roomId: map['room_id'] as int,
        name: map['name'] as String,
        frequencyDays: map['frequency_days'] as int,
        lastCompleted: map['last_completed'] as String?,
        nextDue: map['next_due'] as String,
        notes: (map['notes'] as String?) ?? '',
      );
}

class BudgetCategory {
  final int? id;
  final String name;
  final double monthlyLimit;

  const BudgetCategory({
    this.id,
    required this.name,
    required this.monthlyLimit,
  });

  factory BudgetCategory.fromMap(Map<String, Object?> map) => BudgetCategory(
        id: map['id'] as int?,
        name: map['name'] as String,
        monthlyLimit: (map['monthly_limit'] as num).toDouble(),
      );
}

class Bill {
  final int? id;
  final String name;
  final double amount;
  final int dueDay;
  final int? categoryId;
  final String category;
  final String recurrence;
  final String? nextDue;
  final String notes;

  const Bill({
    this.id,
    required this.name,
    required this.amount,
    required this.dueDay,
    this.categoryId,
    this.category = '',
    this.recurrence = 'Monthly',
    this.nextDue,
    this.notes = '',
  });

  factory Bill.fromMap(Map<String, Object?> map) => Bill(
        id: map['id'] as int?,
        name: map['name'] as String,
        amount: (map['amount'] as num).toDouble(),
        dueDay: map['due_day'] as int,
        categoryId: map['category_id'] as int?,
        category: ((map['category_name'] ?? map['category']) as String?) ?? '',
        recurrence: (map['recurrence'] as String?) ?? 'Monthly',
        nextDue: map['next_due'] as String?,
        notes: (map['notes'] as String?) ?? '',
      );

  DateTime nextOccurrence({DateTime? from}) {
    final now = from ?? DateTime.now();
    final today = DateTime(now.year, now.month, now.day);
    DateTime due;

    if (nextDue != null && nextDue!.isNotEmpty) {
      due = DateTime.parse(nextDue!);
      due = DateTime(due.year, due.month, due.day);
    } else {
      final safeDay = dueDay.clamp(1, 28).toInt();
      due = DateTime(today.year, today.month, safeDay);
    }

    while (due.isBefore(today)) {
      if (recurrence == 'Weekly') {
        due = due.add(const Duration(days: 7));
      } else if (recurrence == 'Fortnightly') {
        due = due.add(const Duration(days: 14));
      } else if (recurrence == 'Yearly') {
        final maxDay = DateTime(due.year + 1, due.month + 1, 0).day;
        due = DateTime(
          due.year + 1,
          due.month,
          dueDay.clamp(1, maxDay).toInt(),
        );
      } else {
        final nextMonth = due.month == 12 ? 1 : due.month + 1;
        final nextYear = due.month == 12 ? due.year + 1 : due.year;
        final maxDay = DateTime(nextYear, nextMonth + 1, 0).day;
        due = DateTime(
          nextYear,
          nextMonth,
          dueDay.clamp(1, maxDay).toInt(),
        );
      }
    }

    return due;
  }

  double monthlyEquivalent() {
    switch (recurrence) {
      case 'Weekly':
        return amount * 52 / 12;
      case 'Fortnightly':
        return amount * 26 / 12;
      case 'Yearly':
        return amount / 12;
      case 'Monthly':
      default:
        return amount;
    }
  }
}

class Project {
  final int? id;
  final String name;
  final int? roomId;
  final double budget;
  final String status;
  final String? startDate;
  final String? targetDate;
  final String notes;
  final String projectType;

  const Project({
    this.id,
    required this.name,
    this.roomId,
    required this.budget,
    required this.status,
    this.startDate,
    this.targetDate,
    this.notes = '',
    this.projectType = 'Home improvement',
  });

  factory Project.fromMap(Map<String, Object?> map) => Project(
        id: map['id'] as int?,
        name: map['name'] as String,
        roomId: map['room_id'] as int?,
        budget: (map['budget'] as num).toDouble(),
        status: map['status'] as String,
        startDate: map['start_date'] as String?,
        targetDate: map['target_date'] as String?,
        notes: (map['notes'] as String?) ?? '',
        projectType: (map['project_type'] as String?) ?? 'Home improvement',
      );
}

class InventoryItem {
  final int? id;
  final String name;
  final int? roomId;
  final String category;
  final double? purchasePrice;
  final String? purchaseDate;
  final String? warrantyExpiry;
  final String serialNumber;
  final String notes;

  const InventoryItem({
    this.id,
    required this.name,
    this.roomId,
    this.category = '',
    this.purchasePrice,
    this.purchaseDate,
    this.warrantyExpiry,
    this.serialNumber = '',
    this.notes = '',
  });

  factory InventoryItem.fromMap(Map<String, Object?> map) => InventoryItem(
        id: map['id'] as int?,
        name: map['name'] as String,
        roomId: map['room_id'] as int?,
        category: (map['category'] as String?) ?? '',
        purchasePrice: map['purchase_price'] == null
            ? null
            : (map['purchase_price'] as num).toDouble(),
        purchaseDate: map['purchase_date'] as String?,
        warrantyExpiry: map['warranty_expiry'] as String?,
        serialNumber: (map['serial_number'] as String?) ?? '',
        notes: (map['notes'] as String?) ?? '',
      );
}


Writing /content/home_organiser/lib/models/app_models.dart


In [6]:
# @title

%%writefile /content/home_organiser/lib/db/app_database.dart
import 'package:path/path.dart';
import 'package:sqflite/sqflite.dart';

import '../models/app_models.dart';

class AppDatabase {
  AppDatabase._();

  static final AppDatabase instance = AppDatabase._();
  Database? _database;

  Future<Database> get database async {
    _database ??= await _openDatabase();
    return _database!;
  }

  Future<Database> _openDatabase() async {
    final dbPath = await getDatabasesPath();

    return openDatabase(
      join(dbPath, 'my_home.db'),
      version: 4,
      onConfigure: (db) async => db.execute('PRAGMA foreign_keys = ON'),
      onCreate: (db, version) async {
        await _createCoreTables(db);
        await _createV2Tables(db);
        await _createV3Tables(db);
        await _createV4Tables(db);
      },
      onUpgrade: (db, oldVersion, newVersion) async {
        if (oldVersion < 2) {
          await _createV2Tables(db);
        }
        if (oldVersion < 3) {
          await _createV3Tables(db);
        }
        if (oldVersion < 4) {
          await _createV4Tables(db);
        }
      },
    );
  }

  Future<void> _createCoreTables(Database db) async {
    await db.execute('''
      CREATE TABLE IF NOT EXISTS rooms (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS chores (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        room_id INTEGER NOT NULL,
        name TEXT NOT NULL,
        frequency_days INTEGER NOT NULL,
        last_completed TEXT,
        next_due TEXT NOT NULL,
        notes TEXT NOT NULL DEFAULT '',
        FOREIGN KEY(room_id) REFERENCES rooms(id) ON DELETE CASCADE
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS chore_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        chore_id INTEGER NOT NULL,
        completed_at TEXT NOT NULL,
        FOREIGN KEY(chore_id) REFERENCES chores(id) ON DELETE CASCADE
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS budget_categories (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL UNIQUE,
        monthly_limit REAL NOT NULL
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS expenses (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        category_id INTEGER NOT NULL,
        amount REAL NOT NULL,
        spent_on TEXT NOT NULL,
        notes TEXT NOT NULL DEFAULT '',
        FOREIGN KEY(category_id) REFERENCES budget_categories(id) ON DELETE CASCADE
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS bills (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        amount REAL NOT NULL,
        due_day INTEGER NOT NULL,
        category TEXT NOT NULL DEFAULT '',
        notes TEXT NOT NULL DEFAULT ''
      )
    ''');
  }

  Future<void> _createV2Tables(Database db) async {
    await db.execute('''
      CREATE TABLE IF NOT EXISTS projects (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        room_id INTEGER,
        budget REAL NOT NULL DEFAULT 0,
        status TEXT NOT NULL DEFAULT 'Active',
        start_date TEXT,
        target_date TEXT,
        notes TEXT NOT NULL DEFAULT '',
        FOREIGN KEY(room_id) REFERENCES rooms(id) ON DELETE SET NULL
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS project_tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        project_id INTEGER NOT NULL,
        name TEXT NOT NULL,
        is_done INTEGER NOT NULL DEFAULT 0,
        FOREIGN KEY(project_id) REFERENCES projects(id) ON DELETE CASCADE
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS project_expenses (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        project_id INTEGER NOT NULL,
        amount REAL NOT NULL,
        spent_on TEXT NOT NULL,
        notes TEXT NOT NULL DEFAULT '',
        FOREIGN KEY(project_id) REFERENCES projects(id) ON DELETE CASCADE
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS inventory_items (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        room_id INTEGER,
        category TEXT NOT NULL DEFAULT '',
        purchase_price REAL,
        purchase_date TEXT,
        warranty_expiry TEXT,
        serial_number TEXT NOT NULL DEFAULT '',
        notes TEXT NOT NULL DEFAULT '',
        FOREIGN KEY(room_id) REFERENCES rooms(id) ON DELETE SET NULL
      )
    ''');
  }

  Future<bool> _columnExists(DatabaseExecutor db, String table, String column) async {
    final info = await db.rawQuery('PRAGMA table_info($table)');
    return info.any((row) => row['name'] == column);
  }

  Future<void> _createV3Tables(Database db) async {
    await db.execute('''
      CREATE TABLE IF NOT EXISTS bill_categories (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL UNIQUE
      )
    ''');

    if (!await _columnExists(db, 'bills', 'category_id')) {
      await db.execute('ALTER TABLE bills ADD COLUMN category_id INTEGER');
    }
    if (!await _columnExists(db, 'bills', 'recurrence')) {
      await db.execute(
        "ALTER TABLE bills ADD COLUMN recurrence TEXT NOT NULL DEFAULT 'Monthly'",
      );
    }
    if (!await _columnExists(db, 'bills', 'next_due')) {
      await db.execute('ALTER TABLE bills ADD COLUMN next_due TEXT');
    }

    await _seedDefaultBillCategories(db);
    await _rebuildBillCategoriesFromBills(db);
    await _populateMissingBillDates(db);
  }

  Future<void> _createV4Tables(Database db) async {
    if (!await _columnExists(db, 'projects', 'project_type')) {
      await db.execute(
        "ALTER TABLE projects ADD COLUMN project_type TEXT NOT NULL DEFAULT 'Home improvement'",
      );
    }
    if (!await _columnExists(db, 'project_tasks', 'due_date')) {
      await db.execute('ALTER TABLE project_tasks ADD COLUMN due_date TEXT');
    }

    await db.execute('''
      CREATE TABLE IF NOT EXISTS app_settings (
        key TEXT PRIMARY KEY,
        value TEXT NOT NULL
      )
    ''');
    await _seedDefaultSettings(db);
  }

  static const Map<String, String> _defaultSettings = {
    'theme_mode': 'system',
    'notifications_enabled': '0',
    'notify_overdue': '1',
    'notify_due_today': '1',
    'notify_pending_projects': '1',
    'notify_upcoming_bills': '1',
    'notification_hour': '9',
    'notification_minute': '0',
  };

  Future<void> _seedDefaultSettings(DatabaseExecutor db) async {
    for (final entry in _defaultSettings.entries) {
      await db.insert(
        'app_settings',
        {'key': entry.key, 'value': entry.value},
        conflictAlgorithm: ConflictAlgorithm.ignore,
      );
    }
  }

  static const _defaultBillCategories = [
    'Council Tax',
    'Mortgage / Rent',
    'Gas',
    'Electricity',
    'Water',
    'Broadband',
    'Mobile',
    'Insurance',
    'TV / Streaming',
    'Subscriptions',
    'Car',
    'Loan / Finance',
    'Other',
  ];

  Future<void> _seedDefaultBillCategories(DatabaseExecutor db) async {
    for (final name in _defaultBillCategories) {
      await db.insert(
        'bill_categories',
        {'name': name},
        conflictAlgorithm: ConflictAlgorithm.ignore,
      );
    }
  }

  Future<void> _rebuildBillCategoriesFromBills(DatabaseExecutor db) async {
    final rows = await db.query('bills');
    for (final bill in rows) {
      final category = ((bill['category'] as String?) ?? '').trim();
      if (category.isEmpty) continue;

      await db.insert(
        'bill_categories',
        {'name': category},
        conflictAlgorithm: ConflictAlgorithm.ignore,
      );
      final categoryRows = await db.query(
        'bill_categories',
        columns: ['id'],
        where: 'name = ?',
        whereArgs: [category],
        limit: 1,
      );
      if (categoryRows.isNotEmpty) {
        await db.update(
          'bills',
          {'category_id': categoryRows.first['id']},
          where: 'id = ? AND category_id IS NULL',
          whereArgs: [bill['id']],
        );
      }
    }
  }

  Future<void> _populateMissingBillDates(DatabaseExecutor db) async {
    final rows = await db.query(
      'bills',
      where: 'next_due IS NULL OR next_due = ?',
      whereArgs: [''],
    );
    final now = DateTime.now();
    final today = DateTime(now.year, now.month, now.day);

    for (final bill in rows) {
      final dueDay = ((bill['due_day'] as num?)?.toInt() ?? 1).clamp(1, 31).toInt();
      final maxThisMonth = DateTime(now.year, now.month + 1, 0).day;
      var due = DateTime(now.year, now.month, dueDay.clamp(1, maxThisMonth).toInt());
      if (due.isBefore(today)) {
        final nextMonth = now.month == 12 ? 1 : now.month + 1;
        final nextYear = now.month == 12 ? now.year + 1 : now.year;
        final maxNextMonth = DateTime(nextYear, nextMonth + 1, 0).day;
        due = DateTime(nextYear, nextMonth, dueDay.clamp(1, maxNextMonth).toInt());
      }
      await db.update(
        'bills',
        {'next_due': dateOnly(due), 'recurrence': 'Monthly'},
        where: 'id = ?',
        whereArgs: [bill['id']],
      );
    }
  }

  String dateOnly(DateTime value) =>
      DateTime(value.year, value.month, value.day)
          .toIso8601String()
          .substring(0, 10);

  // ROOMS

  Future<List<Room>> getRooms() async {
    final db = await database;
    final rows = await db.query('rooms', orderBy: 'name COLLATE NOCASE');
    return rows.map(Room.fromMap).toList();
  }

  Future<int> addRoom(String name) async {
    final db = await database;
    return db.insert('rooms', {'name': name.trim()});
  }

  Future<void> deleteRoom(int id) async {
    final db = await database;
    await db.delete('rooms', where: 'id = ?', whereArgs: [id]);
  }

  // CHORES

  Future<List<Chore>> getChoresForRoom(int roomId) async {
    final db = await database;
    final rows = await db.query(
      'chores',
      where: 'room_id = ?',
      whereArgs: [roomId],
      orderBy: 'next_due ASC, name COLLATE NOCASE',
    );
    return rows.map(Chore.fromMap).toList();
  }

  Future<int> addChore({
    required int roomId,
    required String name,
    required int frequencyDays,
    required DateTime firstDueDate,
    String notes = '',
  }) async {
    final db = await database;
    return db.insert('chores', {
      'room_id': roomId,
      'name': name.trim(),
      'frequency_days': frequencyDays,
      'last_completed': null,
      'next_due': dateOnly(firstDueDate),
      'notes': notes.trim(),
    });
  }

  Future<void> updateChore({
    required int id,
    required String name,
    required int frequencyDays,
    required DateTime nextDueDate,
    required String notes,
  }) async {
    final db = await database;
    await db.update(
      'chores',
      {
        'name': name.trim(),
        'frequency_days': frequencyDays,
        'next_due': dateOnly(nextDueDate),
        'notes': notes.trim(),
      },
      where: 'id = ?',
      whereArgs: [id],
    );
  }

  Future<void> completeChore(Chore chore) async {
    if (chore.id == null) return;

    final db = await database;
    final now = DateTime.now();
    final nextDue = DateTime(now.year, now.month, now.day)
        .add(Duration(days: chore.frequencyDays));

    await db.transaction((txn) async {
      await txn.insert('chore_history', {
        'chore_id': chore.id,
        'completed_at': dateOnly(now),
      });

      await txn.update(
        'chores',
        {
          'last_completed': dateOnly(now),
          'next_due': dateOnly(nextDue),
        },
        where: 'id = ?',
        whereArgs: [chore.id],
      );
    });
  }

  Future<void> deleteChore(int id) async {
    final db = await database;
    await db.delete('chores', where: 'id = ?', whereArgs: [id]);
  }

  Future<List<Map<String, Object?>>> getDueChores() async {
    final db = await database;
    return db.rawQuery('''
      SELECT c.id, c.name, c.next_due, c.frequency_days, r.name AS room_name
      FROM chores c
      JOIN rooms r ON r.id = c.room_id
      WHERE c.next_due <= ?
      ORDER BY c.next_due ASC, r.name COLLATE NOCASE
    ''', [dateOnly(DateTime.now())]);
  }

  Future<List<Map<String, Object?>>> getChoreHistory(int choreId) async {
    final db = await database;
    return db.query(
      'chore_history',
      where: 'chore_id = ?',
      whereArgs: [choreId],
      orderBy: 'completed_at DESC, id DESC',
      limit: 30,
    );
  }

  // BUDGET

  Future<List<BudgetCategory>> getBudgetCategories() async {
    final db = await database;
    final rows =
        await db.query('budget_categories', orderBy: 'name COLLATE NOCASE');
    return rows.map(BudgetCategory.fromMap).toList();
  }

  Future<int> addBudgetCategory(String name, double monthlyLimit) async {
    final db = await database;
    return db.insert('budget_categories', {
      'name': name.trim(),
      'monthly_limit': monthlyLimit,
    });
  }

  Future<int> getExpenseCountForCategory(int categoryId) async {
    final db = await database;
    final result = await db.rawQuery(
      'SELECT COUNT(*) AS count FROM expenses WHERE category_id = ?',
      [categoryId],
    );
    return (result.first['count'] as num).toInt();
  }

  Future<bool> deleteBudgetCategoryIfUnused(int categoryId) async {
    final db = await database;
    return db.transaction((txn) async {
      final result = await txn.rawQuery(
        'SELECT COUNT(*) AS count FROM expenses WHERE category_id = ?',
        [categoryId],
      );
      final count = (result.first['count'] as num).toInt();
      if (count > 0) return false;

      await txn.delete(
        'budget_categories',
        where: 'id = ?',
        whereArgs: [categoryId],
      );
      return true;
    });
  }

  Future<int> addExpense({
    required int categoryId,
    required double amount,
    required String notes,
    DateTime? date,
  }) async {
    final db = await database;
    return db.insert('expenses', {
      'category_id': categoryId,
      'amount': amount,
      'spent_on': dateOnly(date ?? DateTime.now()),
      'notes': notes.trim(),
    });
  }

  Future<void> deleteExpense(int id) async {
    final db = await database;
    await db.delete('expenses', where: 'id = ?', whereArgs: [id]);
  }

  Future<List<Map<String, Object?>>> getBudgetSummaryForCurrentMonth() async {
    final db = await database;
    final now = DateTime.now();
    final start = DateTime(now.year, now.month, 1);
    final end = DateTime(now.year, now.month + 1, 1);

    return db.rawQuery('''
      SELECT b.id, b.name, b.monthly_limit,
             COALESCE(SUM(e.amount), 0) AS spent
      FROM budget_categories b
      LEFT JOIN expenses e
        ON e.category_id = b.id
        AND e.spent_on >= ?
        AND e.spent_on < ?
      GROUP BY b.id, b.name, b.monthly_limit
      ORDER BY b.name COLLATE NOCASE
    ''', [dateOnly(start), dateOnly(end)]);
  }

  Future<double> getCurrentMonthExpenseTotal() async {
    final db = await database;
    final now = DateTime.now();
    final start = dateOnly(DateTime(now.year, now.month, 1));
    final end = dateOnly(DateTime(now.year, now.month + 1, 1));

    final result = await db.rawQuery('''
      SELECT COALESCE(SUM(amount), 0) AS total
      FROM expenses
      WHERE spent_on >= ? AND spent_on < ?
    ''', [start, end]);

    return (result.first['total'] as num).toDouble();
  }

  Future<List<Map<String, Object?>>> getRecentExpenses() async {
    final db = await database;
    return db.rawQuery('''
      SELECT e.id, e.amount, e.spent_on, e.notes,
             b.name AS category_name
      FROM expenses e
      JOIN budget_categories b ON b.id = e.category_id
      ORDER BY e.spent_on DESC, e.id DESC
      LIMIT 40
    ''');
  }

  // BILLS

  Future<List<Map<String, Object?>>> getBillCategories() async {
    final db = await database;
    return db.query('bill_categories', orderBy: 'name COLLATE NOCASE');
  }

  Future<int> addBillCategory(String name) async {
    final db = await database;
    return db.insert(
      'bill_categories',
      {'name': name.trim()},
      conflictAlgorithm: ConflictAlgorithm.ignore,
    );
  }

  Future<bool> deleteBillCategoryIfUnused(int categoryId) async {
    final db = await database;
    return db.transaction((txn) async {
      final result = await txn.rawQuery(
        'SELECT COUNT(*) AS count FROM bills WHERE category_id = ?',
        [categoryId],
      );
      final count = (result.first['count'] as num).toInt();
      if (count > 0) return false;
      await txn.delete(
        'bill_categories',
        where: 'id = ?',
        whereArgs: [categoryId],
      );
      return true;
    });
  }

  Future<List<Bill>> getBills() async {
    final db = await database;
    final rows = await db.rawQuery('''
      SELECT b.*, c.name AS category_name
      FROM bills b
      LEFT JOIN bill_categories c ON c.id = b.category_id
      ORDER BY b.next_due ASC, b.due_day ASC, b.name COLLATE NOCASE
    ''');
    return rows.map(Bill.fromMap).toList();
  }

  Future<int> addBill({
    required String name,
    required double amount,
    required int dueDay,
    int? categoryId,
    String categoryName = '',
    String recurrence = 'Monthly',
    required String nextDue,
    String notes = '',
  }) async {
    final db = await database;
    return db.insert('bills', {
      'name': name.trim(),
      'amount': amount,
      'due_day': dueDay,
      'category': categoryName.trim(),
      'category_id': categoryId,
      'recurrence': recurrence,
      'next_due': nextDue,
      'notes': notes.trim(),
    });
  }

  Future<void> deleteBill(int id) async {
    final db = await database;
    await db.delete('bills', where: 'id = ?', whereArgs: [id]);
  }

  // PROJECTS

  Future<List<Map<String, Object?>>> getProjects() async {
    final db = await database;
    return db.rawQuery('''
      SELECT
        p.*,
        r.name AS room_name,
        COALESCE(SUM(pe.amount), 0) AS spent,
        (SELECT COUNT(*) FROM project_tasks pt WHERE pt.project_id = p.id) AS task_count,
        (SELECT COUNT(*) FROM project_tasks pt WHERE pt.project_id = p.id AND pt.is_done = 1) AS done_count
      FROM projects p
      LEFT JOIN rooms r ON r.id = p.room_id
      LEFT JOIN project_expenses pe ON pe.project_id = p.id
      GROUP BY p.id
      ORDER BY
        CASE WHEN p.status = 'Completed' THEN 1 ELSE 0 END,
        p.id DESC
    ''');
  }

  Future<int> addProject({
    required String name,
    int? roomId,
    required double budget,
    String projectType = 'Home improvement',
    String? targetDate,
    String notes = '',
  }) async {
    final db = await database;
    return db.insert('projects', {
      'name': name.trim(),
      'room_id': roomId,
      'budget': budget,
      'project_type': projectType,
      'status': 'Active',
      'start_date': dateOnly(DateTime.now()),
      'target_date': targetDate,
      'notes': notes.trim(),
    });
  }

  Future<void> setProjectStatus(int id, String status) async {
    final db = await database;
    await db.update(
      'projects',
      {'status': status},
      where: 'id = ?',
      whereArgs: [id],
    );
  }

  Future<void> deleteProject(int id) async {
    final db = await database;
    await db.delete('projects', where: 'id = ?', whereArgs: [id]);
  }

  Future<List<Map<String, Object?>>> getProjectTasks(int projectId) async {
    final db = await database;
    return db.query(
      'project_tasks',
      where: 'project_id = ?',
      whereArgs: [projectId],
      orderBy: 'is_done ASC, id ASC',
    );
  }

  Future<int> addProjectTask(int projectId, String name, {String? dueDate}) async {
    final db = await database;
    return db.insert('project_tasks', {
      'project_id': projectId,
      'name': name.trim(),
      'is_done': 0,
      'due_date': dueDate,
    });
  }

  Future<void> setProjectTaskDone(int id, bool done) async {
    final db = await database;
    await db.update(
      'project_tasks',
      {'is_done': done ? 1 : 0},
      where: 'id = ?',
      whereArgs: [id],
    );
  }

  Future<void> deleteProjectTask(int id) async {
    final db = await database;
    await db.delete('project_tasks', where: 'id = ?', whereArgs: [id]);
  }

  Future<int> addProjectExpense({
    required int projectId,
    required double amount,
    required String notes,
  }) async {
    final db = await database;
    return db.insert('project_expenses', {
      'project_id': projectId,
      'amount': amount,
      'spent_on': dateOnly(DateTime.now()),
      'notes': notes.trim(),
    });
  }

  Future<List<Map<String, Object?>>> getProjectExpenses(int projectId) async {
    final db = await database;
    return db.query(
      'project_expenses',
      where: 'project_id = ?',
      whereArgs: [projectId],
      orderBy: 'spent_on DESC, id DESC',
    );
  }

  Future<int> getActiveProjectCount() async {
    final db = await database;
    final result = await db.rawQuery(
      "SELECT COUNT(*) AS count FROM projects WHERE status != 'Completed'",
    );
    return (result.first['count'] as num).toInt();
  }


  // SETTINGS / REMINDER COUNTS

  Future<Map<String, String>> getSettings() async {
    final db = await database;
    await _seedDefaultSettings(db);
    final rows = await db.query('app_settings');
    return {
      for (final row in rows) row['key'] as String: row['value'] as String,
    };
  }

  Future<String> getSetting(String key, {String fallback = ''}) async {
    final db = await database;
    final rows = await db.query(
      'app_settings',
      columns: ['value'],
      where: 'key = ?',
      whereArgs: [key],
      limit: 1,
    );
    if (rows.isEmpty) return fallback;
    return (rows.first['value'] as String?) ?? fallback;
  }

  Future<void> setSetting(String key, String value) async {
    final db = await database;
    await db.insert(
      'app_settings',
      {'key': key, 'value': value},
      conflictAlgorithm: ConflictAlgorithm.replace,
    );
  }

  Future<Map<String, int>> getReminderCounts() async {
    final db = await database;
    final today = dateOnly(DateTime.now());

    Future<int> q(String sql, [List<Object?>? args]) async {
      final rows = await db.rawQuery(sql, args);
      return (rows.first['count'] as num).toInt();
    }

    final overdueChores = await q(
      'SELECT COUNT(*) AS count FROM chores WHERE next_due < ?',
      [today],
    );
    final dueChores = await q(
      'SELECT COUNT(*) AS count FROM chores WHERE next_due = ?',
      [today],
    );
    final overdueProjectTasks = await q(
      'SELECT COUNT(*) AS count FROM project_tasks WHERE is_done = 0 AND due_date IS NOT NULL AND due_date < ?',
      [today],
    );
    final dueProjectTasks = await q(
      'SELECT COUNT(*) AS count FROM project_tasks WHERE is_done = 0 AND due_date = ?',
      [today],
    );
    final pendingProjectTasks = await q(
      'SELECT COUNT(*) AS count FROM project_tasks WHERE is_done = 0 AND (due_date IS NULL OR due_date > ?)',
      [today],
    );

    final now = DateTime.now();
    final start = DateTime(now.year, now.month, now.day);
    final cutoff = start.add(const Duration(days: 3));
    final bills = await getBills();
    final upcomingBills = bills.where((bill) {
      final next = bill.nextOccurrence(from: start);
      return !next.isAfter(cutoff);
    }).length;

    return {
      'overdue': overdueChores + overdueProjectTasks,
      'due_today': dueChores + dueProjectTasks,
      'pending_projects': pendingProjectTasks,
      'upcoming_bills': upcomingBills,
    };
  }

  // INVENTORY

  Future<List<Map<String, Object?>>> getInventoryItems() async {
    final db = await database;
    return db.rawQuery('''
      SELECT i.*, r.name AS room_name
      FROM inventory_items i
      LEFT JOIN rooms r ON r.id = i.room_id
      ORDER BY i.name COLLATE NOCASE
    ''');
  }

  Future<int> addInventoryItem({
    required String name,
    int? roomId,
    String category = '',
    double? purchasePrice,
    String? purchaseDate,
    String? warrantyExpiry,
    String serialNumber = '',
    String notes = '',
  }) async {
    final db = await database;
    return db.insert('inventory_items', {
      'name': name.trim(),
      'room_id': roomId,
      'category': category.trim(),
      'purchase_price': purchasePrice,
      'purchase_date': purchaseDate,
      'warranty_expiry': warrantyExpiry,
      'serial_number': serialNumber.trim(),
      'notes': notes.trim(),
    });
  }

  Future<void> deleteInventoryItem(int id) async {
    final db = await database;
    await db.delete('inventory_items', where: 'id = ?', whereArgs: [id]);
  }

  Future<int> getInventoryCount() async {
    final db = await database;
    final result =
        await db.rawQuery('SELECT COUNT(*) AS count FROM inventory_items');
    return (result.first['count'] as num).toInt();
  }

  // BACKUP / RESTORE

  static const backupTables = [
    'rooms',
    'chores',
    'chore_history',
    'budget_categories',
    'expenses',
    'bill_categories',
    'bills',
    'projects',
    'project_tasks',
    'project_expenses',
    'inventory_items',
    'app_settings',
  ];

  Future<Map<String, dynamic>> exportAllData() async {
    final db = await database;
    final tables = <String, dynamic>{};

    for (final table in backupTables) {
      tables[table] = await db.query(table);
    }

    return {
      'app': 'MyHome',
      'schemaVersion': 4,
      'exportedAt': DateTime.now().toIso8601String(),
      'tables': tables,
    };
  }

  Future<Set<String>> _tableColumns(DatabaseExecutor db, String table) async {
    final info = await db.rawQuery('PRAGMA table_info($table)');
    return info.map((row) => row['name'] as String).toSet();
  }

  Future<void> restoreAllData(Map<String, dynamic> backup) async {
    if (backup['app'] != 'MyHome') {
      throw const FormatException('This is not a My Home backup file.');
    }

    final rawTables = backup['tables'];
    if (rawTables is! Map) {
      throw const FormatException('Backup tables are missing.');
    }

    final db = await database;

    const deleteOrder = [
      'app_settings',
      'chore_history',
      'chores',
      'project_tasks',
      'project_expenses',
      'inventory_items',
      'expenses',
      'bills',
      'bill_categories',
      'budget_categories',
      'projects',
      'rooms',
    ];

    const insertOrder = [
      'rooms',
      'budget_categories',
      'bill_categories',
      'bills',
      'chores',
      'chore_history',
      'expenses',
      'projects',
      'project_tasks',
      'project_expenses',
      'inventory_items',
      'app_settings',
    ];

    await db.transaction((txn) async {
      for (final table in deleteOrder) {
        await txn.delete(table);
      }

      for (final table in insertOrder) {
        final rows = rawTables[table];
        if (rows is! List) continue;

        final allowedColumns = await _tableColumns(txn, table);
        for (final rawRow in rows) {
          if (rawRow is! Map) continue;
          final row = <String, Object?>{};
          rawRow.forEach((key, value) {
            final column = key.toString();
            if (allowedColumns.contains(column)) {
              row[column] = value as Object?;
            }
          });

          if (row.isEmpty) continue;
          await txn.insert(
            table,
            row,
            conflictAlgorithm: ConflictAlgorithm.replace,
          );
        }
      }

      // Older backups had bill categories stored only as free text and did
      // not include recurrence or next_due. Rebuild those newer fields after
      // the old rows have been restored.
      final hasBillCategoryBackup = rawTables['bill_categories'] is List;
      if (!hasBillCategoryBackup) {
        await _seedDefaultBillCategories(txn);
      }
      await _rebuildBillCategoriesFromBills(txn);
      await _populateMissingBillDates(txn);
      await _seedDefaultSettings(txn);
    });
  }

}


Writing /content/home_organiser/lib/db/app_database.dart


In [7]:
# @title

%%writefile /content/home_organiser/lib/services/backup_service.dart
import 'dart:convert';
import 'dart:typed_data';

import 'package:file_picker/file_picker.dart';

import '../db/app_database.dart';

class BackupService {
  static Future<bool> saveBackup() async {
    final data = await AppDatabase.instance.exportAllData();
    final jsonText = const JsonEncoder.withIndent('  ').convert(data);
    final bytes = Uint8List.fromList(utf8.encode(jsonText));

    final now = DateTime.now();
    String two(int value) => value.toString().padLeft(2, '0');

    final name =
        'my_home_backup_${now.year}-${two(now.month)}-${two(now.day)}_'
        '${two(now.hour)}${two(now.minute)}.json';

    final result = await FilePicker.saveFile(
      dialogTitle: 'Save My Home backup',
      fileName: name,
      bytes: bytes,
    );

    return result != null;
  }

  static Future<bool> restoreBackup() async {
    final file = await FilePicker.pickFile(
      type: FileType.custom,
      allowedExtensions: ['json'],
    );

    if (file == null) return false;

    final bytes = await file.readAsBytes();
    final decoded = jsonDecode(utf8.decode(bytes));

    if (decoded is! Map) {
      throw const FormatException('Invalid backup file.');
    }

    await AppDatabase.instance.restoreAllData(
      Map<String, dynamic>.from(decoded),
    );

    return true;
  }
}


Writing /content/home_organiser/lib/services/backup_service.dart


In [8]:
# @title
%%writefile /content/home_organiser/lib/screens/dashboard_screen.dart
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../db/app_database.dart';
import '../models/app_models.dart';

class DashboardScreen extends StatefulWidget {
  const DashboardScreen({super.key});

  @override
  State<DashboardScreen> createState() => _DashboardScreenState();
}

class _DashboardScreenState extends State<DashboardScreen> {
  Future<Map<String, Object>> _load() async {
    return {
      'due': await AppDatabase.instance.getDueChores(),
      'spending': await AppDatabase.instance.getCurrentMonthExpenseTotal(),
      'bills': await AppDatabase.instance.getBills(),
      'projects': await AppDatabase.instance.getActiveProjectCount(),
      'inventory': await AppDatabase.instance.getInventoryCount(),
    };
  }

  @override
  Widget build(BuildContext context) {
    final money = NumberFormat.currency(locale: 'en_GB', symbol: '£');

    return Scaffold(
      appBar: AppBar(title: const Text('My Home')),
      body: FutureBuilder<Map<String, Object>>(
        future: _load(),
        builder: (context, snapshot) {
          if (!snapshot.hasData) {
            return const Center(child: CircularProgressIndicator());
          }

          final due = snapshot.data!['due'] as List<Map<String, Object?>>;
          final spending = snapshot.data!['spending'] as double;
          final bills = snapshot.data!['bills'] as List<Bill>;
          final projects = snapshot.data!['projects'] as int;
          final inventory = snapshot.data!['inventory'] as int;

          final now = DateTime.now();
          final today = DateTime(now.year, now.month, now.day);
          final todayText = today.toIso8601String().substring(0, 10);
          final overdue = due.where((row) {
            return (row['next_due'] as String).compareTo(todayText) < 0;
          }).length;

          final cutoff = today.add(const Duration(days: 30));
          final upcomingBills = bills
              .map((bill) => MapEntry(bill, bill.nextOccurrence(from: today)))
              .where((entry) => !entry.value.isAfter(cutoff))
              .toList()
            ..sort((a, b) => a.value.compareTo(b.value));

          return RefreshIndicator(
            onRefresh: () async => setState(() {}),
            child: ListView(
              padding: const EdgeInsets.fromLTRB(16, 16, 16, 100),
              children: [
                Text(
                  DateFormat('EEEE d MMMM').format(now),
                  style: Theme.of(context).textTheme.titleMedium,
                ),
                const SizedBox(height: 16),
                Wrap(
                  spacing: 10,
                  runSpacing: 10,
                  children: [
                    _Card(
                      icon: Icons.cleaning_services_outlined,
                      title: 'Chores due',
                      value: '${due.length}',
                      subtitle: '$overdue overdue',
                    ),
                    _Card(
                      icon: Icons.account_balance_wallet_outlined,
                      title: 'Spent',
                      value: money.format(spending),
                      subtitle: 'this month',
                    ),
                    _Card(
                      icon: Icons.receipt_long_outlined,
                      title: 'Bills',
                      value: '${upcomingBills.length}',
                      subtitle: 'next 30 days',
                    ),
                    _Card(
                      icon: Icons.handyman_outlined,
                      title: 'Projects',
                      value: '$projects',
                      subtitle: 'active',
                    ),
                    _Card(
                      icon: Icons.inventory_2_outlined,
                      title: 'Inventory',
                      value: '$inventory',
                      subtitle: 'items',
                    ),
                  ],
                ),
                const SizedBox(height: 24),
                Text(
                  'Upcoming bills',
                  style: Theme.of(context).textTheme.titleLarge,
                ),
                const SizedBox(height: 8),
                if (upcomingBills.isEmpty)
                  const Card(
                    child: ListTile(
                      leading: Icon(Icons.event_available_outlined),
                      title: Text('No bills due in the next 30 days'),
                    ),
                  )
                else
                  ...upcomingBills.take(6).map((entry) {
                    final bill = entry.key;
                    final dueDate = entry.value;
                    return Card(
                      child: ListTile(
                        leading: const Icon(Icons.receipt_long_outlined),
                        title: Text(bill.name),
                        subtitle: Text(
                          '${bill.category.isEmpty ? bill.recurrence : bill.category} • '
                          'due ${DateFormat('EEE d MMM').format(dueDate)}',
                        ),
                        trailing: Text(money.format(bill.amount)),
                      ),
                    );
                  }),
                const SizedBox(height: 24),
                Text(
                  'Chores needing attention',
                  style: Theme.of(context).textTheme.titleLarge,
                ),
                const SizedBox(height: 8),
                if (due.isEmpty)
                  const Card(
                    child: ListTile(
                      leading: Icon(Icons.check_circle_outline),
                      title: Text('Nothing is due'),
                    ),
                  )
                else
                  ...due.take(10).map((row) {
                    final date = DateTime.parse(row['next_due'] as String);
                    return Card(
                      child: ListTile(
                        leading: const Icon(Icons.today_outlined),
                        title: Text(row['name'] as String),
                        subtitle: Text(
                          '${row['room_name']} • due ${DateFormat('d MMM').format(date)}',
                        ),
                      ),
                    );
                  }),
              ],
            ),
          );
        },
      ),
    );
  }
}

class _Card extends StatelessWidget {
  final IconData icon;
  final String title;
  final String value;
  final String subtitle;

  const _Card({
    required this.icon,
    required this.title,
    required this.value,
    required this.subtitle,
  });

  @override
  Widget build(BuildContext context) {
    return SizedBox(
      width: 160,
      child: Card(
        child: Padding(
          padding: const EdgeInsets.all(14),
          child: Column(
            crossAxisAlignment: CrossAxisAlignment.start,
            children: [
              Icon(icon),
              const SizedBox(height: 10),
              Text(title),
              Text(value, style: Theme.of(context).textTheme.headlineSmall),
              Text(subtitle, style: Theme.of(context).textTheme.bodySmall),
            ],
          ),
        ),
      ),
    );
  }
}


Writing /content/home_organiser/lib/screens/dashboard_screen.dart


In [9]:
# @title

%%writefile /content/home_organiser/lib/screens/rooms_screen.dart
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../db/app_database.dart';
import '../models/app_models.dart';

class RoomsScreen extends StatefulWidget {
  const RoomsScreen({super.key});

  @override
  State<RoomsScreen> createState() => _RoomsScreenState();
}

class _RoomsScreenState extends State<RoomsScreen> {
  Future<void> _addRoom() async {
    final controller = TextEditingController();

    final name = await showDialog<String>(
      context: context,
      builder: (context) => AlertDialog(
        title: const Text('Add room'),
        content: TextField(
          controller: controller,
          autofocus: true,
          textCapitalization: TextCapitalization.words,
          decoration: const InputDecoration(labelText: 'Room name'),
        ),
        actions: [
          TextButton(
            onPressed: () => Navigator.pop(context),
            child: const Text('Cancel'),
          ),
          FilledButton(
            onPressed: () => Navigator.pop(context, controller.text.trim()),
            child: const Text('Add'),
          ),
        ],
      ),
    );

    if (name == null || name.isEmpty) return;
    await AppDatabase.instance.addRoom(name);
    if (mounted) setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Rooms')),
      floatingActionButton: FloatingActionButton.extended(
        onPressed: _addRoom,
        icon: const Icon(Icons.add),
        label: const Text('Room'),
      ),
      body: FutureBuilder<List<Room>>(
        future: AppDatabase.instance.getRooms(),
        builder: (context, snapshot) {
          if (!snapshot.hasData) {
            return const Center(child: CircularProgressIndicator());
          }

          final rooms = snapshot.data!;

          if (rooms.isEmpty) {
            return const Center(
              child: Text('Add your first room, then add chores to it.'),
            );
          }

          return ListView.separated(
            padding: const EdgeInsets.fromLTRB(12, 12, 12, 100),
            itemCount: rooms.length,
            separatorBuilder: (_, _) => const SizedBox(height: 4),
            itemBuilder: (context, index) {
              final room = rooms[index];
              return Card(
                child: ListTile(
                  leading: const CircleAvatar(
                    child: Icon(Icons.meeting_room_outlined),
                  ),
                  title: Text(room.name),
                  subtitle: const Text('View recurring chores'),
                  trailing: const Icon(Icons.chevron_right),
                  onTap: () async {
                    await Navigator.push(
                      context,
                      MaterialPageRoute(
                        builder: (_) => RoomDetailScreen(room: room),
                      ),
                    );
                    if (mounted) setState(() {});
                  },
                ),
              );
            },
          );
        },
      ),
    );
  }
}

class RoomDetailScreen extends StatefulWidget {
  final Room room;

  const RoomDetailScreen({super.key, required this.room});

  @override
  State<RoomDetailScreen> createState() => _RoomDetailScreenState();
}

class _RoomDetailScreenState extends State<RoomDetailScreen> {
  bool _isDue(Chore chore) {
    final now = DateTime.now();
    final today = DateTime(now.year, now.month, now.day);
    return !DateTime.parse(chore.nextDue).isAfter(today);
  }

  Future<void> _choreDialog({Chore? chore}) async {
    final name = TextEditingController(text: chore?.name ?? '');
    final days = TextEditingController(
      text: (chore?.frequencyDays ?? 7).toString(),
    );
    final notes = TextEditingController(text: chore?.notes ?? '');
    DateTime dueDate = chore == null
        ? DateTime.now()
        : DateTime.parse(chore.nextDue);

    final result = await showDialog<Map<String, Object>>(
      context: context,
      builder: (context) => StatefulBuilder(
        builder: (context, setDialogState) => AlertDialog(
          title: Text(chore == null ? 'Add chore' : 'Edit chore'),
          content: SingleChildScrollView(
            child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                TextField(
                  controller: name,
                  autofocus: true,
                  decoration: const InputDecoration(labelText: 'Chore'),
                ),
                const SizedBox(height: 12),
                TextField(
                  controller: days,
                  keyboardType: TextInputType.number,
                  decoration: const InputDecoration(
                    labelText: 'Repeat every (days)',
                    helperText: '1 daily • 7 weekly • 14 fortnightly • 30 monthly-ish',
                  ),
                ),
                const SizedBox(height: 12),
                ListTile(
                  contentPadding: EdgeInsets.zero,
                  leading: const Icon(Icons.event_outlined),
                  title: Text(chore == null ? 'First due date' : 'Next due date'),
                  subtitle: Text(DateFormat('d MMM yyyy').format(dueDate)),
                  trailing: const Icon(Icons.calendar_month_outlined),
                  onTap: () async {
                    final picked = await showDatePicker(
                      context: context,
                      initialDate: dueDate,
                      firstDate: DateTime(2000),
                      lastDate: DateTime(2100),
                    );
                    if (picked != null) {
                      setDialogState(() => dueDate = picked);
                    }
                  },
                ),
                const SizedBox(height: 12),
                TextField(
                  controller: notes,
                  maxLines: 3,
                  decoration: const InputDecoration(labelText: 'Notes'),
                ),
              ],
            ),
          ),
          actions: [
            TextButton(
              onPressed: () => Navigator.pop(context),
              child: const Text('Cancel'),
            ),
            FilledButton(
              onPressed: () {
                final parsedDays = int.tryParse(days.text.trim()) ?? 0;
                if (name.text.trim().isEmpty || parsedDays < 1) {
                  return;
                }
                Navigator.pop(context, {
                  'name': name.text.trim(),
                  'days': parsedDays,
                  'dueDate': dueDate,
                  'notes': notes.text.trim(),
                });
              },
              child: Text(chore == null ? 'Add' : 'Save'),
            ),
          ],
        ),
      ),
    );

    if (result == null) {
      return;
    }

    if (chore == null) {
      await AppDatabase.instance.addChore(
        roomId: widget.room.id!,
        name: result['name'] as String,
        frequencyDays: result['days'] as int,
        firstDueDate: result['dueDate'] as DateTime,
        notes: result['notes'] as String,
      );
    } else {
      await AppDatabase.instance.updateChore(
        id: chore.id!,
        name: result['name'] as String,
        frequencyDays: result['days'] as int,
        nextDueDate: result['dueDate'] as DateTime,
        notes: result['notes'] as String,
      );
    }

    if (mounted) {
      setState(() {});
    }
  }

  Future<void> _showHistory(Chore chore) async {
    final history = await AppDatabase.instance.getChoreHistory(chore.id!);
    if (!mounted) return;

    await showModalBottomSheet(
      context: context,
      showDragHandle: true,
      builder: (context) => SafeArea(
        child: Padding(
          padding: const EdgeInsets.fromLTRB(20, 0, 20, 24),
          child: Column(
            mainAxisSize: MainAxisSize.min,
            crossAxisAlignment: CrossAxisAlignment.stretch,
            children: [
              Text(chore.name, style: Theme.of(context).textTheme.titleLarge),
              if (chore.notes.isNotEmpty) ...[
                const SizedBox(height: 6),
                Text(chore.notes),
              ],
              const SizedBox(height: 16),
              const Text('Completion history'),
              if (history.isEmpty)
                const Padding(
                  padding: EdgeInsets.only(top: 8),
                  child: Text('No completions recorded yet.'),
                )
              else
                ...history.map(
                  (row) => ListTile(
                    contentPadding: EdgeInsets.zero,
                    leading: const Icon(Icons.check_circle_outline),
                    title: Text(
                      DateFormat('d MMMM yyyy').format(
                        DateTime.parse(row['completed_at'] as String),
                      ),
                    ),
                  ),
                ),
            ],
          ),
        ),
      ),
    );
  }

  Future<void> _deleteRoom() async {
    final yes = await showDialog<bool>(
      context: context,
      builder: (context) => AlertDialog(
        title: Text('Delete ${widget.room.name}?'),
        content: const Text('Its chores and chore history will also be deleted.'),
        actions: [
          TextButton(
            onPressed: () => Navigator.pop(context, false),
            child: const Text('Cancel'),
          ),
          FilledButton(
            onPressed: () => Navigator.pop(context, true),
            child: const Text('Delete'),
          ),
        ],
      ),
    );

    if (yes == true) {
      await AppDatabase.instance.deleteRoom(widget.room.id!);
      if (mounted) Navigator.pop(context);
    }
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(
        title: Text(widget.room.name),
        actions: [
          IconButton(
            onPressed: _deleteRoom,
            icon: const Icon(Icons.delete_outline),
          ),
        ],
      ),
      floatingActionButton: FloatingActionButton.extended(
        onPressed: () => _choreDialog(),
        icon: const Icon(Icons.add_task),
        label: const Text('Chore'),
      ),
      body: FutureBuilder<List<Chore>>(
        future: AppDatabase.instance.getChoresForRoom(widget.room.id!),
        builder: (context, snapshot) {
          if (!snapshot.hasData) {
            return const Center(child: CircularProgressIndicator());
          }

          final chores = snapshot.data!;
          final dueCount = chores.where(_isDue).length;
          final score =
              chores.isEmpty ? 1.0 : (chores.length - dueCount) / chores.length;

          return ListView(
            padding: const EdgeInsets.fromLTRB(16, 16, 16, 100),
            children: [
              Card(
                child: Padding(
                  padding: const EdgeInsets.all(16),
                  child: Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      Row(
                        children: [
                          const Text('Room status'),
                          const Spacer(),
                          Text('${(score * 100).round()}%'),
                        ],
                      ),
                      const SizedBox(height: 10),
                      LinearProgressIndicator(value: score),
                      const SizedBox(height: 6),
                      Text('$dueCount of ${chores.length} chores currently due'),
                    ],
                  ),
                ),
              ),
              const SizedBox(height: 12),
              if (chores.isEmpty)
                const Card(child: ListTile(title: Text('No chores yet')))
              else
                ...chores.map((chore) {
                  final due = DateTime.parse(chore.nextDue);
                  final isDue = _isDue(chore);

                  return Card(
                    child: ListTile(
                      onTap: () => _showHistory(chore),
                      leading: Icon(
                        isDue
                            ? Icons.radio_button_unchecked
                            : Icons.check_circle_outline,
                      ),
                      title: Text(chore.name),
                      subtitle: Text(
                        'Every ${chore.frequencyDays} days • '
                        '${isDue ? 'Due' : 'Next'} ${DateFormat('d MMM').format(due)}',
                      ),
                      trailing: Row(
                        mainAxisSize: MainAxisSize.min,
                        children: [
                          IconButton(
                            tooltip: 'Complete',
                            onPressed: () async {
                              await AppDatabase.instance.completeChore(chore);
                              if (mounted) setState(() {});
                            },
                            icon: const Icon(Icons.done),
                          ),
                          PopupMenuButton<String>(
                            onSelected: (value) async {
                              if (value == 'edit') {
                                await _choreDialog(chore: chore);
                              } else if (value == 'delete') {
                                await AppDatabase.instance.deleteChore(chore.id!);
                                if (mounted) setState(() {});
                              }
                            },
                            itemBuilder: (_) => const [
                              PopupMenuItem(value: 'edit', child: Text('Edit')),
                              PopupMenuItem(value: 'delete', child: Text('Delete')),
                            ],
                          ),
                        ],
                      ),
                    ),
                  );
                }),
            ],
          );
        },
      ),
    );
  }
}


Writing /content/home_organiser/lib/screens/rooms_screen.dart


In [10]:
# @title

%%writefile /content/home_organiser/lib/screens/budget_screen.dart
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../db/app_database.dart';
import '../models/app_models.dart';

class BudgetScreen extends StatefulWidget {
  const BudgetScreen({super.key});

  @override
  State<BudgetScreen> createState() => _BudgetScreenState();
}

class _BudgetScreenState extends State<BudgetScreen> {
  Future<Map<String, Object>> _load() async => {
        'summary': await AppDatabase.instance.getBudgetSummaryForCurrentMonth(),
        'categories': await AppDatabase.instance.getBudgetCategories(),
        'expenses': await AppDatabase.instance.getRecentExpenses(),
      };

  Future<void> _addCategory() async {
    final name = TextEditingController();
    final amount = TextEditingController();

    final result = await showDialog<Map<String, Object>>(
      context: context,
      builder: (context) => AlertDialog(
        title: const Text('Add budget category'),
        content: Column(
          mainAxisSize: MainAxisSize.min,
          children: [
            TextField(
              controller: name,
              decoration: const InputDecoration(labelText: 'Category'),
            ),
            const SizedBox(height: 12),
            TextField(
              controller: amount,
              keyboardType:
                  const TextInputType.numberWithOptions(decimal: true),
              decoration: const InputDecoration(
                labelText: 'Monthly budget',
                prefixText: '£',
              ),
            ),
          ],
        ),
        actions: [
          TextButton(
            onPressed: () => Navigator.pop(context),
            child: const Text('Cancel'),
          ),
          FilledButton(
            onPressed: () {
              final parsed = double.tryParse(amount.text.trim());
              if (name.text.trim().isEmpty || parsed == null || parsed < 0) {
                return;
              }
              Navigator.pop(context, {
                'name': name.text.trim(),
                'amount': parsed,
              });
            },
            child: const Text('Add'),
          ),
        ],
      ),
    );

    if (result == null) return;

    try {
      await AppDatabase.instance.addBudgetCategory(
        result['name'] as String,
        result['amount'] as double,
      );
    } catch (_) {
      if (mounted) {
        ScaffoldMessenger.of(context).showSnackBar(
          const SnackBar(content: Text('That category already exists.')),
        );
      }
    }

    if (mounted) setState(() {});
  }

  Future<void> _deleteCategory({
    required int categoryId,
    required String categoryName,
  }) async {
    final expenseCount =
        await AppDatabase.instance.getExpenseCountForCategory(categoryId);
    if (!mounted) return;

    if (expenseCount > 0) {
      ScaffoldMessenger.of(context).showSnackBar(
        SnackBar(
          content: Text(
            '$categoryName cannot be removed because it has '
            '$expenseCount saved ${expenseCount == 1 ? 'expense' : 'expenses'}. '
            'Delete those expenses first.',
          ),
        ),
      );
      return;
    }

    final confirmed = await showDialog<bool>(
      context: context,
      builder: (context) => AlertDialog(
        title: const Text('Remove category?'),
        content: Text('Remove the category "$categoryName"?'),
        actions: [
          TextButton(
            onPressed: () => Navigator.pop(context, false),
            child: const Text('Cancel'),
          ),
          FilledButton(
            onPressed: () => Navigator.pop(context, true),
            child: const Text('Remove'),
          ),
        ],
      ),
    );

    if (confirmed != true) return;

    final removed =
        await AppDatabase.instance.deleteBudgetCategoryIfUnused(categoryId);
    if (!mounted) return;

    if (!removed) {
      ScaffoldMessenger.of(context).showSnackBar(
        const SnackBar(
          content: Text(
            'That category now has saved expenses, so it was not removed.',
          ),
        ),
      );
      return;
    }

    setState(() {});
  }

  Future<void> _addExpense(List<BudgetCategory> categories) async {
    if (categories.isEmpty) {
      ScaffoldMessenger.of(context).showSnackBar(
        const SnackBar(content: Text('Add a budget category first.')),
      );
      return;
    }

    int selected = categories.first.id!;
    final amount = TextEditingController();
    final notes = TextEditingController();

    final result = await showDialog<Map<String, Object>>(
      context: context,
      builder: (context) => StatefulBuilder(
        builder: (context, setDialogState) => AlertDialog(
          title: const Text('Add expense'),
          content: Column(
            mainAxisSize: MainAxisSize.min,
            children: [
              DropdownButtonFormField<int>(
                initialValue: selected,
                decoration: const InputDecoration(labelText: 'Category'),
                items: categories
                    .map(
                      (c) => DropdownMenuItem(
                        value: c.id!,
                        child: Text(c.name),
                      ),
                    )
                    .toList(),
                onChanged: (value) {
                  if (value != null) {
                    setDialogState(() => selected = value);
                  }
                },
              ),
              const SizedBox(height: 12),
              TextField(
                controller: amount,
                keyboardType:
                    const TextInputType.numberWithOptions(decimal: true),
                decoration: const InputDecoration(
                  labelText: 'Amount',
                  prefixText: '£',
                ),
              ),
              const SizedBox(height: 12),
              TextField(
                controller: notes,
                decoration: const InputDecoration(labelText: 'Description'),
              ),
            ],
          ),
          actions: [
            TextButton(
              onPressed: () => Navigator.pop(context),
              child: const Text('Cancel'),
            ),
            FilledButton(
              onPressed: () {
                final parsed = double.tryParse(amount.text.trim());
                if (parsed == null || parsed <= 0) return;
                Navigator.pop(context, {
                  'categoryId': selected,
                  'amount': parsed,
                  'notes': notes.text.trim(),
                });
              },
              child: const Text('Add'),
            ),
          ],
        ),
      ),
    );

    if (result == null) return;

    await AppDatabase.instance.addExpense(
      categoryId: result['categoryId'] as int,
      amount: result['amount'] as double,
      notes: result['notes'] as String,
    );
    if (mounted) setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final money = NumberFormat.currency(locale: 'en_GB', symbol: '£');

    return Scaffold(
      appBar: AppBar(title: const Text('Money')),
      body: FutureBuilder<Map<String, Object>>(
        future: _load(),
        builder: (context, snapshot) {
          if (!snapshot.hasData) {
            return const Center(child: CircularProgressIndicator());
          }

          final summary =
              snapshot.data!['summary'] as List<Map<String, Object?>>;
          final categories =
              snapshot.data!['categories'] as List<BudgetCategory>;
          final expenses =
              snapshot.data!['expenses'] as List<Map<String, Object?>>;

          final totalBudget = summary.fold<double>(
            0,
            (sum, row) => sum + (row['monthly_limit'] as num).toDouble(),
          );
          final totalSpent = summary.fold<double>(
            0,
            (sum, row) => sum + (row['spent'] as num).toDouble(),
          );

          return ListView(
            padding: const EdgeInsets.fromLTRB(16, 16, 16, 100),
            children: [
              Card(
                child: Padding(
                  padding: const EdgeInsets.all(16),
                  child: Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      Text(DateFormat('MMMM yyyy').format(DateTime.now())),
                      const SizedBox(height: 8),
                      Text(
                        money.format(totalSpent),
                        style: Theme.of(context).textTheme.headlineMedium,
                      ),
                      Text('of ${money.format(totalBudget)} budgeted'),
                      const SizedBox(height: 10),
                      LinearProgressIndicator(
                        value: totalBudget <= 0
                            ? 0
                            : (totalSpent / totalBudget)
                                .clamp(0.0, 1.0)
                                .toDouble(),
                      ),
                    ],
                  ),
                ),
              ),
              const SizedBox(height: 10),
              Row(
                children: [
                  Expanded(
                    child: FilledButton.icon(
                      onPressed: _addCategory,
                      icon: const Icon(Icons.add),
                      label: const Text('Category'),
                    ),
                  ),
                  const SizedBox(width: 8),
                  Expanded(
                    child: FilledButton.tonalIcon(
                      onPressed: () => _addExpense(categories),
                      icon: const Icon(Icons.add_card),
                      label: const Text('Expense'),
                    ),
                  ),
                ],
              ),
              const SizedBox(height: 22),
              Text('Categories', style: Theme.of(context).textTheme.titleLarge),
              const SizedBox(height: 8),
              ...summary.map((row) {
                final limit = (row['monthly_limit'] as num).toDouble();
                final spent = (row['spent'] as num).toDouble();
                return Card(
                  child: Padding(
                    padding: const EdgeInsets.all(14),
                    child: Column(
                      crossAxisAlignment: CrossAxisAlignment.start,
                      children: [
                        Row(
                          children: [
                            Expanded(child: Text(row['name'] as String)),
                            Text(money.format(limit - spent)),
                            IconButton(
                              tooltip: 'Remove category',
                              onPressed: () => _deleteCategory(
                                categoryId: row['id'] as int,
                                categoryName: row['name'] as String,
                              ),
                              icon: const Icon(Icons.delete_outline),
                            ),
                          ],
                        ),
                        const SizedBox(height: 8),
                        LinearProgressIndicator(
                          value: limit <= 0
                              ? 0
                              : (spent / limit).clamp(0.0, 1.0).toDouble(),
                        ),
                        const SizedBox(height: 6),
                        Text('${money.format(spent)} of ${money.format(limit)}'),
                      ],
                    ),
                  ),
                );
              }),
              const SizedBox(height: 22),
              Text(
                'Recent expenses',
                style: Theme.of(context).textTheme.titleLarge,
              ),
              ...expenses.map(
                (row) => ListTile(
                  contentPadding: EdgeInsets.zero,
                  title: Text(
                    (row['notes'] as String?)?.isNotEmpty == true
                        ? row['notes'] as String
                        : row['category_name'] as String,
                  ),
                  subtitle: Text(
                    '${row['category_name']} • '
                    '${DateFormat('d MMM').format(DateTime.parse(row['spent_on'] as String))}',
                  ),
                  trailing: Row(
                    mainAxisSize: MainAxisSize.min,
                    children: [
                      Text(money.format((row['amount'] as num).toDouble())),
                      IconButton(
                        tooltip: 'Delete',
                        onPressed: () async {
                          await AppDatabase.instance
                              .deleteExpense(row['id'] as int);
                          if (mounted) setState(() {});
                        },
                        icon: const Icon(Icons.delete_outline),
                      ),
                    ],
                  ),
                ),
              ),
            ],
          );
        },
      ),
    );
  }
}


Writing /content/home_organiser/lib/screens/budget_screen.dart


In [11]:
# @title
%%writefile /content/home_organiser/lib/screens/bills_screen.dart
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../db/app_database.dart';
import '../models/app_models.dart';

class BillsScreen extends StatefulWidget {
  const BillsScreen({super.key});

  @override
  State<BillsScreen> createState() => _BillsScreenState();
}

class _BillsScreenState extends State<BillsScreen> {
  static const _recurrences = [
    'Monthly',
    'Weekly',
    'Fortnightly',
    'Yearly',
  ];

  Future<void> _addBill() async {
    final categories = await AppDatabase.instance.getBillCategories();
    if (!mounted) return;

    final name = TextEditingController();
    final amount = TextEditingController();
    final notes = TextEditingController();
    int? categoryId;
    var recurrence = 'Monthly';
    var nextDue = DateTime.now();

    final result = await showDialog<Map<String, Object?>>(
      context: context,
      builder: (context) => StatefulBuilder(
        builder: (context, setDialogState) => AlertDialog(
          title: const Text('Add recurring bill'),
          content: SingleChildScrollView(
            child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                TextField(
                  controller: name,
                  decoration: const InputDecoration(labelText: 'Bill name'),
                ),
                const SizedBox(height: 10),
                TextField(
                  controller: amount,
                  keyboardType:
                      const TextInputType.numberWithOptions(decimal: true),
                  decoration:
                      const InputDecoration(labelText: 'Amount', prefixText: '£'),
                ),
                const SizedBox(height: 10),
                DropdownButtonFormField<int?>(
                  initialValue: categoryId,
                  decoration: const InputDecoration(labelText: 'Category'),
                  items: [
                    const DropdownMenuItem<int?>(
                      value: null,
                      child: Text('Uncategorised'),
                    ),
                    ...categories.map(
                      (category) => DropdownMenuItem<int?>(
                        value: category['id'] as int,
                        child: Text(category['name'] as String),
                      ),
                    ),
                  ],
                  onChanged: (value) =>
                      setDialogState(() => categoryId = value),
                ),
                const SizedBox(height: 10),
                DropdownButtonFormField<String>(
                  initialValue: recurrence,
                  decoration: const InputDecoration(labelText: 'Repeats'),
                  items: _recurrences
                      .map(
                        (value) => DropdownMenuItem(
                          value: value,
                          child: Text(value),
                        ),
                      )
                      .toList(),
                  onChanged: (value) {
                    if (value != null) {
                      setDialogState(() => recurrence = value);
                    }
                  },
                ),
                const SizedBox(height: 10),
                ListTile(
                  contentPadding: EdgeInsets.zero,
                  title: const Text('Next payment'),
                  subtitle: Text(DateFormat('EEE d MMM yyyy').format(nextDue)),
                  trailing: const Icon(Icons.calendar_month_outlined),
                  onTap: () async {
                    final picked = await showDatePicker(
                      context: context,
                      initialDate: nextDue,
                      firstDate: DateTime(2000),
                      lastDate: DateTime(2100),
                    );
                    if (picked != null) {
                      setDialogState(() => nextDue = picked);
                    }
                  },
                ),
                const SizedBox(height: 10),
                TextField(
                  controller: notes,
                  decoration: const InputDecoration(labelText: 'Notes'),
                ),
              ],
            ),
          ),
          actions: [
            TextButton(
              onPressed: () => Navigator.pop(context),
              child: const Text('Cancel'),
            ),
            FilledButton(
              onPressed: () {
                final parsedAmount = double.tryParse(amount.text.trim());
                if (name.text.trim().isEmpty ||
                    parsedAmount == null ||
                    parsedAmount < 0) {
                  return;
                }

                String categoryName = '';
                if (categoryId != null) {
                  final match = categories.firstWhere(
                    (item) => item['id'] == categoryId,
                  );
                  categoryName = match['name'] as String;
                }

                Navigator.pop(context, {
                  'name': name.text.trim(),
                  'amount': parsedAmount,
                  'categoryId': categoryId,
                  'categoryName': categoryName,
                  'recurrence': recurrence,
                  'nextDue': AppDatabase.instance.dateOnly(nextDue),
                  'notes': notes.text.trim(),
                });
              },
              child: const Text('Add'),
            ),
          ],
        ),
      ),
    );

    if (result == null) return;

    final nextDueText = result['nextDue'] as String;
    final nextDueDate = DateTime.parse(nextDueText);
    await AppDatabase.instance.addBill(
      name: result['name'] as String,
      amount: result['amount'] as double,
      dueDay: nextDueDate.day,
      categoryId: result['categoryId'] as int?,
      categoryName: result['categoryName'] as String,
      recurrence: result['recurrence'] as String,
      nextDue: nextDueText,
      notes: result['notes'] as String,
    );
    if (mounted) setState(() {});
  }

  Future<void> _manageCategories() async {
    final controller = TextEditingController();
    var categories = await AppDatabase.instance.getBillCategories();
    if (!mounted) return;

    await showDialog<void>(
      context: context,
      builder: (context) => StatefulBuilder(
        builder: (context, setDialogState) => AlertDialog(
          title: const Text('Bill categories'),
          content: SizedBox(
            width: 420,
            child: SingleChildScrollView(
              child: Column(
                mainAxisSize: MainAxisSize.min,
                children: [
                  Row(
                    children: [
                      Expanded(
                        child: TextField(
                          controller: controller,
                          decoration:
                              const InputDecoration(labelText: 'New category'),
                        ),
                      ),
                      const SizedBox(width: 8),
                      IconButton.filled(
                        tooltip: 'Add category',
                        onPressed: () async {
                          final value = controller.text.trim();
                          if (value.isEmpty) return;
                          await AppDatabase.instance.addBillCategory(value);
                          controller.clear();
                          categories =
                              await AppDatabase.instance.getBillCategories();
                          setDialogState(() {});
                        },
                        icon: const Icon(Icons.add),
                      ),
                    ],
                  ),
                  const SizedBox(height: 12),
                  ...categories.map(
                    (category) => ListTile(
                      contentPadding: EdgeInsets.zero,
                      title: Text(category['name'] as String),
                      trailing: IconButton(
                        tooltip: 'Delete category',
                        icon: const Icon(Icons.delete_outline),
                        onPressed: () async {
                          final removed = await AppDatabase.instance
                              .deleteBillCategoryIfUnused(category['id'] as int);
                          if (!context.mounted) return;
                          if (!removed) {
                            ScaffoldMessenger.of(context).showSnackBar(
                              const SnackBar(
                                content: Text(
                                  'This category is being used by a bill. '
                                  'Delete or change that bill first.',
                                ),
                              ),
                            );
                            return;
                          }
                          categories =
                              await AppDatabase.instance.getBillCategories();
                          setDialogState(() {});
                        },
                      ),
                    ),
                  ),
                ],
              ),
            ),
          ),
          actions: [
            TextButton(
              onPressed: () => Navigator.pop(context),
              child: const Text('Done'),
            ),
          ],
        ),
      ),
    );

    if (mounted) setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final money = NumberFormat.currency(locale: 'en_GB', symbol: '£');

    return Scaffold(
      appBar: AppBar(
        title: const Text('Bills'),
        actions: [
          IconButton(
            tooltip: 'Manage bill categories',
            onPressed: _manageCategories,
            icon: const Icon(Icons.category_outlined),
          ),
        ],
      ),
      floatingActionButton: FloatingActionButton.extended(
        onPressed: _addBill,
        icon: const Icon(Icons.add),
        label: const Text('Bill'),
      ),
      body: FutureBuilder<List<Bill>>(
        future: AppDatabase.instance.getBills(),
        builder: (context, snapshot) {
          if (!snapshot.hasData) {
            return const Center(child: CircularProgressIndicator());
          }

          final bills = snapshot.data!;
          final monthlyTotal = bills.fold<double>(
            0,
            (sum, bill) => sum + bill.monthlyEquivalent(),
          );
          final today = DateTime.now();
          final ordered = [...bills]
            ..sort(
              (a, b) => a
                  .nextOccurrence(from: today)
                  .compareTo(b.nextOccurrence(from: today)),
            );

          return RefreshIndicator(
            onRefresh: () async => setState(() {}),
            child: ListView(
              padding: const EdgeInsets.fromLTRB(16, 16, 16, 100),
              children: [
                Card(
                  child: ListTile(
                    leading: const Icon(Icons.receipt_long_outlined),
                    title: Text(
                      money.format(monthlyTotal),
                      style: Theme.of(context).textTheme.headlineSmall,
                    ),
                    subtitle: const Text('Approximate monthly bill total'),
                  ),
                ),
                const SizedBox(height: 10),
                if (ordered.isEmpty)
                  const Card(child: ListTile(title: Text('No bills saved')))
                else
                  ...ordered.map((bill) {
                    final next = bill.nextOccurrence(from: today);
                    return Card(
                      child: ListTile(
                        leading: CircleAvatar(child: Text('${next.day}')),
                        title: Text(bill.name),
                        subtitle: Text(
                          '${bill.category.isEmpty ? 'Uncategorised' : bill.category} • '
                          '${bill.recurrence.toLowerCase()} • '
                          'next ${DateFormat('d MMM yyyy').format(next)}',
                        ),
                        trailing: Row(
                          mainAxisSize: MainAxisSize.min,
                          children: [
                            Text(money.format(bill.amount)),
                            IconButton(
                              tooltip: 'Delete bill',
                              onPressed: () async {
                                await AppDatabase.instance.deleteBill(bill.id!);
                                if (mounted) setState(() {});
                              },
                              icon: const Icon(Icons.delete_outline),
                            ),
                          ],
                        ),
                      ),
                    );
                  }),
              ],
            ),
          );
        },
      ),
    );
  }
}


Writing /content/home_organiser/lib/screens/bills_screen.dart


In [12]:
# @title

%%writefile /content/home_organiser/lib/screens/projects_screen.dart
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../db/app_database.dart';
import '../models/app_models.dart';

class ProjectsScreen extends StatefulWidget {
  const ProjectsScreen({super.key});

  @override
  State<ProjectsScreen> createState() => _ProjectsScreenState();
}

class _ProjectsScreenState extends State<ProjectsScreen> {
  Future<void> _addProject() async {
    final rooms = await AppDatabase.instance.getRooms();
    if (!mounted) {
      return;
    }

    final name = TextEditingController();
    final budget = TextEditingController(text: '0');
    final notes = TextEditingController();
    int? roomId;
    String projectType = 'Home improvement';

    final result = await showDialog<Map<String, Object?>>(
      context: context,
      builder: (context) => StatefulBuilder(
        builder: (context, setDialogState) => AlertDialog(
          title: const Text('Add project'),
          content: SingleChildScrollView(
            child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                TextField(
                  controller: name,
                  autofocus: true,
                  decoration: const InputDecoration(labelText: 'Project name'),
                ),
                const SizedBox(height: 10),
                DropdownButtonFormField<String>(
                  initialValue: projectType,
                  decoration: const InputDecoration(labelText: 'Project type'),
                  items: const [
                    DropdownMenuItem(value: 'Home improvement', child: Text('Home improvement')),
                    DropdownMenuItem(value: 'Learning / skill', child: Text('Learning / skill')),
                    DropdownMenuItem(value: 'Personal', child: Text('Personal')),
                    DropdownMenuItem(value: 'Fitness / health', child: Text('Fitness / health')),
                    DropdownMenuItem(value: 'Maintenance', child: Text('Maintenance')),
                    DropdownMenuItem(value: 'Other', child: Text('Other')),
                  ],
                  onChanged: (value) {
                    if (value != null) {
                      setDialogState(() => projectType = value);
                    }
                  },
                ),
                const SizedBox(height: 10),
                DropdownButtonFormField<int?>(
                  initialValue: roomId,
                  decoration:
                      const InputDecoration(labelText: 'Room (optional)'),
                  items: [
                    const DropdownMenuItem<int?>(
                      value: null,
                      child: Text('Whole home / no room'),
                    ),
                    ...rooms.map(
                      (room) => DropdownMenuItem<int?>(
                        value: room.id,
                        child: Text(room.name),
                      ),
                    ),
                  ],
                  onChanged: (value) =>
                      setDialogState(() => roomId = value),
                ),
                const SizedBox(height: 10),
                TextField(
                  controller: budget,
                  keyboardType:
                      const TextInputType.numberWithOptions(decimal: true),
                  decoration: const InputDecoration(
                    labelText: 'Budget',
                    prefixText: '£',
                  ),
                ),
                const SizedBox(height: 10),
                TextField(
                  controller: notes,
                  maxLines: 3,
                  decoration: const InputDecoration(labelText: 'Notes'),
                ),
              ],
            ),
          ),
          actions: [
            TextButton(
              onPressed: () => Navigator.pop(context),
              child: const Text('Cancel'),
            ),
            FilledButton(
              onPressed: () {
                final parsed = double.tryParse(budget.text.trim()) ?? 0;
                if (name.text.trim().isEmpty || parsed < 0) {
                  return;
                }
                Navigator.pop(context, {
                  'name': name.text.trim(),
                  'roomId': roomId,
                  'projectType': projectType,
                  'budget': parsed,
                  'notes': notes.text.trim(),
                });
              },
              child: const Text('Add'),
            ),
          ],
        ),
      ),
    );

    if (result == null) {
      return;
    }

    await AppDatabase.instance.addProject(
      name: result['name'] as String,
      roomId: result['roomId'] as int?,
      projectType: result['projectType'] as String,
      budget: result['budget'] as double,
      notes: result['notes'] as String,
    );
    if (mounted) {
      setState(() {});
    }
  }

  @override
  Widget build(BuildContext context) {
    final money = NumberFormat.currency(locale: 'en_GB', symbol: '£');

    return Scaffold(
      appBar: AppBar(title: const Text('Projects')),
      floatingActionButton: FloatingActionButton.extended(
        onPressed: _addProject,
        icon: const Icon(Icons.add),
        label: const Text('Project'),
      ),
      body: FutureBuilder<List<Map<String, Object?>>>(
        future: AppDatabase.instance.getProjects(),
        builder: (context, snapshot) {
          if (!snapshot.hasData) {
            return const Center(child: CircularProgressIndicator());
          }

          final projects = snapshot.data!;

          if (projects.isEmpty) {
            return const Center(
              child: Text('Add a decorating, repair or renovation project.'),
            );
          }

          return ListView.builder(
            padding: const EdgeInsets.fromLTRB(12, 12, 12, 100),
            itemCount: projects.length,
            itemBuilder: (context, index) {
              final row = projects[index];
              final spent = (row['spent'] as num).toDouble();
              final budget = (row['budget'] as num).toDouble();
              final taskCount = (row['task_count'] as num).toInt();
              final doneCount = (row['done_count'] as num).toInt();
              final status = row['status'] as String;

              return Card(
                child: ListTile(
                  leading: CircleAvatar(
                    child: Icon(
                      status == 'Completed'
                          ? Icons.check
                          : (row['project_type'] == 'Learning / skill'
                              ? Icons.school_outlined
                              : row['project_type'] == 'Fitness / health'
                                  ? Icons.fitness_center
                                  : row['project_type'] == 'Personal'
                                      ? Icons.person_outline
                                      : Icons.handyman_outlined),
                    ),
                  ),
                  title: Text(row['name'] as String),
                  subtitle: Text(
                    '${row['project_type'] ?? 'Home improvement'} • ${row['room_name'] ?? 'No room'} • '
                    '$doneCount/$taskCount tasks • '
                    '${money.format(spent)} of ${money.format(budget)}',
                  ),
                  trailing: const Icon(Icons.chevron_right),
                  onTap: () async {
                    final project = Project.fromMap(row);
                    await Navigator.push(
                      context,
                      MaterialPageRoute(
                        builder: (_) => ProjectDetailScreen(project: project),
                      ),
                    );
                    if (mounted) {
      setState(() {});
    }
                  },
                ),
              );
            },
          );
        },
      ),
    );
  }
}

class ProjectDetailScreen extends StatefulWidget {
  final Project project;

  const ProjectDetailScreen({super.key, required this.project});

  @override
  State<ProjectDetailScreen> createState() => _ProjectDetailScreenState();
}

class _ProjectDetailScreenState extends State<ProjectDetailScreen> {
  Future<Map<String, Object>> _load() async {
    return {
      'tasks': await AppDatabase.instance.getProjectTasks(widget.project.id!),
      'expenses':
          await AppDatabase.instance.getProjectExpenses(widget.project.id!),
    };
  }

  Future<void> _addTask() async {
    final controller = TextEditingController();
    DateTime? dueDate;

    final result = await showDialog<Map<String, Object?>>(
      context: context,
      builder: (context) => StatefulBuilder(
        builder: (context, setDialogState) => AlertDialog(
          title: const Text('Add project task'),
          content: Column(
            mainAxisSize: MainAxisSize.min,
            children: [
              TextField(
                controller: controller,
                autofocus: true,
                decoration: const InputDecoration(labelText: 'Task'),
              ),
              const SizedBox(height: 10),
              ListTile(
                contentPadding: EdgeInsets.zero,
                leading: const Icon(Icons.event_outlined),
                title: const Text('Due date'),
                subtitle: Text(
                  dueDate == null
                      ? 'No due date — tap to choose'
                      : DateFormat('d MMM yyyy').format(dueDate!),
                ),
                trailing: dueDate == null
                    ? const Icon(Icons.chevron_right)
                    : IconButton(
                        tooltip: 'Remove due date',
                        onPressed: () => setDialogState(() => dueDate = null),
                        icon: const Icon(Icons.clear),
                      ),
                onTap: () async {
                  final picked = await showDatePicker(
                    context: context,
                    initialDate: dueDate ?? DateTime.now(),
                    firstDate: DateTime(2000),
                    lastDate: DateTime(2100),
                  );
                  if (picked != null) {
                    setDialogState(() => dueDate = picked);
                  }
                },
              ),
            ],
          ),
          actions: [
            TextButton(
              onPressed: () => Navigator.pop(context),
              child: const Text('Cancel'),
            ),
            FilledButton(
              onPressed: () {
                final name = controller.text.trim();
                if (name.isEmpty) {
                  return;
                }
                Navigator.pop(context, {'name': name, 'dueDate': dueDate});
              },
              child: const Text('Add'),
            ),
          ],
        ),
      ),
    );

    if (result == null) {
      return;
    }
    final due = result['dueDate'] as DateTime?;
    await AppDatabase.instance.addProjectTask(
      widget.project.id!,
      result['name'] as String,
      dueDate: due == null ? null : AppDatabase.instance.dateOnly(due),
    );
    if (mounted) {
      setState(() {});
    }
  }

  Future<void> _addExpense() async {
    final amount = TextEditingController();
    final notes = TextEditingController();

    final result = await showDialog<Map<String, Object>>(
      context: context,
      builder: (context) => AlertDialog(
        title: const Text('Add project expense'),
        content: Column(
          mainAxisSize: MainAxisSize.min,
          children: [
            TextField(
              controller: amount,
              keyboardType:
                  const TextInputType.numberWithOptions(decimal: true),
              decoration:
                  const InputDecoration(labelText: 'Amount', prefixText: '£'),
            ),
            const SizedBox(height: 10),
            TextField(
              controller: notes,
              decoration: const InputDecoration(labelText: 'Description'),
            ),
          ],
        ),
        actions: [
          TextButton(
            onPressed: () => Navigator.pop(context),
            child: const Text('Cancel'),
          ),
          FilledButton(
            onPressed: () {
              final parsed = double.tryParse(amount.text.trim());
              if (parsed == null || parsed <= 0) {
                return;
              }
              Navigator.pop(context, {
                'amount': parsed,
                'notes': notes.text.trim(),
              });
            },
            child: const Text('Add'),
          ),
        ],
      ),
    );

    if (result == null) {
      return;
    }
    await AppDatabase.instance.addProjectExpense(
      projectId: widget.project.id!,
      amount: result['amount'] as double,
      notes: result['notes'] as String,
    );
    if (mounted) {
      setState(() {});
    }
  }

  Future<void> _deleteProject() async {
    final yes = await showDialog<bool>(
      context: context,
      builder: (context) => AlertDialog(
        title: const Text('Delete project?'),
        content: const Text('Tasks and project expenses will also be deleted.'),
        actions: [
          TextButton(
            onPressed: () => Navigator.pop(context, false),
            child: const Text('Cancel'),
          ),
          FilledButton(
            onPressed: () => Navigator.pop(context, true),
            child: const Text('Delete'),
          ),
        ],
      ),
    );

    if (!mounted) {
      return;
    }

    if (yes == true) {
      final navigator = Navigator.of(context);
      await AppDatabase.instance.deleteProject(widget.project.id!);
      if (!mounted) {
      return;
    }
      navigator.pop();
    }
  }

  Widget? _taskDueText(String? dueText, bool isDone) {
    if (dueText == null || dueText.isEmpty) {
      return null;
    }
    final due = DateTime.parse(dueText);
    final now = DateTime.now();
    final today = DateTime(now.year, now.month, now.day);
    final date = DateTime(due.year, due.month, due.day);
    final formatted = DateFormat('d MMM yyyy').format(date);
    if (isDone) {
      return Text('Completed • due $formatted');
    }
    if (date.isBefore(today)) {
      return Text('Overdue • due $formatted');
    }
    if (date == today) {
      return Text('Due today • $formatted');
    }
    return Text('Due $formatted');
  }

  @override
  Widget build(BuildContext context) {
    final money = NumberFormat.currency(locale: 'en_GB', symbol: '£');

    return Scaffold(
      appBar: AppBar(
        title: Text(widget.project.name),
        actions: [
          IconButton(
            onPressed: _deleteProject,
            icon: const Icon(Icons.delete_outline),
          ),
        ],
      ),
      body: FutureBuilder<Map<String, Object>>(
        future: _load(),
        builder: (context, snapshot) {
          if (!snapshot.hasData) {
            return const Center(child: CircularProgressIndicator());
          }

          final tasks =
              snapshot.data!['tasks'] as List<Map<String, Object?>>;
          final expenses =
              snapshot.data!['expenses'] as List<Map<String, Object?>>;
          final spent = expenses.fold<double>(
            0,
            (sum, row) => sum + (row['amount'] as num).toDouble(),
          );
          final done = tasks
              .where((row) => (row['is_done'] as num).toInt() == 1)
              .length;

          return ListView(
            padding: const EdgeInsets.fromLTRB(16, 16, 16, 100),
            children: [
              Card(
                child: Padding(
                  padding: const EdgeInsets.all(16),
                  child: Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      Row(
                        children: [
                          Expanded(
                            child: Text(
                              '${money.format(spent)} spent',
                              style: Theme.of(context).textTheme.titleLarge,
                            ),
                          ),
                          Text('Budget ${money.format(widget.project.budget)}'),
                        ],
                      ),
                      const SizedBox(height: 10),
                      LinearProgressIndicator(
                        value: widget.project.budget <= 0
                            ? 0
                            : (spent / widget.project.budget)
                                .clamp(0.0, 1.0)
                                .toDouble(),
                      ),
                      const SizedBox(height: 10),
                      Text('$done of ${tasks.length} tasks complete'),
                      const SizedBox(height: 6),
                      Text('Type: ${widget.project.projectType}'),
                      if (widget.project.notes.isNotEmpty) ...[
                        const SizedBox(height: 8),
                        Text(widget.project.notes),
                      ],
                    ],
                  ),
                ),
              ),
              const SizedBox(height: 10),
              Row(
                children: [
                  Expanded(
                    child: FilledButton.icon(
                      onPressed: _addTask,
                      icon: const Icon(Icons.add_task),
                      label: const Text('Task'),
                    ),
                  ),
                  const SizedBox(width: 8),
                  Expanded(
                    child: FilledButton.tonalIcon(
                      onPressed: _addExpense,
                      icon: const Icon(Icons.add_card),
                      label: const Text('Expense'),
                    ),
                  ),
                ],
              ),
              const SizedBox(height: 10),
              FilledButton.tonal(
                onPressed: () async {
                  final navigator = Navigator.of(context);
                  final newStatus = widget.project.status == 'Completed'
                      ? 'Active'
                      : 'Completed';

                  await AppDatabase.instance
                      .setProjectStatus(widget.project.id!, newStatus);

                  navigator.pop();
                },
                child: Text(
                  widget.project.status == 'Completed'
                      ? 'Reopen project'
                      : 'Mark project complete',
                ),
              ),
              const SizedBox(height: 22),
              Text('Tasks', style: Theme.of(context).textTheme.titleLarge),
              ...(tasks.isEmpty
                  ? <Widget>[const Text('No tasks added yet.')]
                  : tasks.map<Widget>((row) {
                  final id = row['id'] as int;
                  final isDone = (row['is_done'] as num).toInt() == 1;

                  return CheckboxListTile(
                    contentPadding: EdgeInsets.zero,
                    value: isDone,
                    title: Text(row['name'] as String),
                    subtitle: _taskDueText(row['due_date'] as String?, isDone),
                    secondary: IconButton(
                      onPressed: () async {
                        await AppDatabase.instance.deleteProjectTask(id);
                        if (mounted) {
      setState(() {});
    }
                      },
                      icon: const Icon(Icons.delete_outline),
                    ),
                    onChanged: (value) async {
                      await AppDatabase.instance
                          .setProjectTaskDone(id, value ?? false);
                      if (mounted) {
      setState(() {});
    }
                    },
                  );
                }).toList()),
              const SizedBox(height: 22),
              Text('Expenses', style: Theme.of(context).textTheme.titleLarge),
              ...(expenses.isEmpty
                  ? <Widget>[const Text('No project expenses yet.')]
                  : expenses.map<Widget>(
                  (row) => ListTile(
                    contentPadding: EdgeInsets.zero,
                    title: Text(
                      (row['notes'] as String?)?.isNotEmpty == true
                          ? row['notes'] as String
                          : 'Project expense',
                    ),
                    subtitle: Text(
                      DateFormat('d MMM yyyy').format(
                        DateTime.parse(row['spent_on'] as String),
                      ),
                    ),
                    trailing:
                        Text(money.format((row['amount'] as num).toDouble())),
                  ),
                ).toList()),
            ],
          );
        },
      ),
    );
  }
}


Writing /content/home_organiser/lib/screens/projects_screen.dart


In [13]:
# @title

%%writefile /content/home_organiser/lib/screens/inventory_screen.dart
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../db/app_database.dart';

class InventoryScreen extends StatefulWidget {
  const InventoryScreen({super.key});

  @override
  State<InventoryScreen> createState() => _InventoryScreenState();
}

class _InventoryScreenState extends State<InventoryScreen> {
  Future<void> _addItem() async {
    final rooms = await AppDatabase.instance.getRooms();
    if (!mounted) return;

    final name = TextEditingController();
    final category = TextEditingController();
    final price = TextEditingController();
    final serial = TextEditingController();
    final notes = TextEditingController();
    int? roomId;
    DateTime? purchaseDate;
    DateTime? warrantyExpiry;

    final result = await showDialog<Map<String, Object?>>(
      context: context,
      builder: (context) => StatefulBuilder(
        builder: (context, setDialogState) => AlertDialog(
          title: const Text('Add inventory item'),
          content: SingleChildScrollView(
            child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                TextField(
                  controller: name,
                  autofocus: true,
                  decoration: const InputDecoration(labelText: 'Item name'),
                ),
                const SizedBox(height: 10),
                DropdownButtonFormField<int?>(
                  initialValue: roomId,
                  decoration:
                      const InputDecoration(labelText: 'Room (optional)'),
                  items: [
                    const DropdownMenuItem<int?>(
                      value: null,
                      child: Text('No room'),
                    ),
                    ...rooms.map(
                      (room) => DropdownMenuItem<int?>(
                        value: room.id,
                        child: Text(room.name),
                      ),
                    ),
                  ],
                  onChanged: (value) =>
                      setDialogState(() => roomId = value),
                ),
                const SizedBox(height: 10),
                TextField(
                  controller: category,
                  decoration: const InputDecoration(
                    labelText: 'Category',
                    hintText: 'Appliance',
                  ),
                ),
                const SizedBox(height: 10),
                TextField(
                  controller: price,
                  keyboardType:
                      const TextInputType.numberWithOptions(decimal: true),
                  decoration: const InputDecoration(
                    labelText: 'Purchase price',
                    prefixText: '£',
                  ),
                ),
                const SizedBox(height: 10),
                ListTile(
                  contentPadding: EdgeInsets.zero,
                  title: const Text('Purchase date'),
                  subtitle: Text(
                    purchaseDate == null
                        ? 'Not set'
                        : DateFormat('d MMM yyyy').format(purchaseDate!),
                  ),
                  trailing: const Icon(Icons.calendar_month),
                  onTap: () async {
                    final picked = await showDatePicker(
                      context: context,
                      firstDate: DateTime(1990),
                      lastDate: DateTime(2100),
                      initialDate: purchaseDate ?? DateTime.now(),
                    );
                    if (picked != null) {
                      setDialogState(() => purchaseDate = picked);
                    }
                  },
                ),
                ListTile(
                  contentPadding: EdgeInsets.zero,
                  title: const Text('Warranty expiry'),
                  subtitle: Text(
                    warrantyExpiry == null
                        ? 'Not set'
                        : DateFormat('d MMM yyyy').format(warrantyExpiry!),
                  ),
                  trailing: const Icon(Icons.event_available_outlined),
                  onTap: () async {
                    final picked = await showDatePicker(
                      context: context,
                      firstDate: DateTime(1990),
                      lastDate: DateTime(2100),
                      initialDate: warrantyExpiry ?? DateTime.now(),
                    );
                    if (picked != null) {
                      setDialogState(() => warrantyExpiry = picked);
                    }
                  },
                ),
                TextField(
                  controller: serial,
                  decoration:
                      const InputDecoration(labelText: 'Serial number'),
                ),
                const SizedBox(height: 10),
                TextField(
                  controller: notes,
                  maxLines: 3,
                  decoration: const InputDecoration(labelText: 'Notes'),
                ),
              ],
            ),
          ),
          actions: [
            TextButton(
              onPressed: () => Navigator.pop(context),
              child: const Text('Cancel'),
            ),
            FilledButton(
              onPressed: () {
                if (name.text.trim().isEmpty) return;
                Navigator.pop(context, {
                  'name': name.text.trim(),
                  'roomId': roomId,
                  'category': category.text.trim(),
                  'price': double.tryParse(price.text.trim()),
                  'purchaseDate': purchaseDate == null
                      ? null
                      : AppDatabase.instance.dateOnly(purchaseDate!),
                  'warrantyExpiry': warrantyExpiry == null
                      ? null
                      : AppDatabase.instance.dateOnly(warrantyExpiry!),
                  'serial': serial.text.trim(),
                  'notes': notes.text.trim(),
                });
              },
              child: const Text('Add'),
            ),
          ],
        ),
      ),
    );

    if (result == null) return;

    await AppDatabase.instance.addInventoryItem(
      name: result['name'] as String,
      roomId: result['roomId'] as int?,
      category: result['category'] as String,
      purchasePrice: result['price'] as double?,
      purchaseDate: result['purchaseDate'] as String?,
      warrantyExpiry: result['warrantyExpiry'] as String?,
      serialNumber: result['serial'] as String,
      notes: result['notes'] as String,
    );

    if (mounted) setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final money = NumberFormat.currency(locale: 'en_GB', symbol: '£');

    return Scaffold(
      appBar: AppBar(title: const Text('Inventory')),
      floatingActionButton: FloatingActionButton.extended(
        onPressed: _addItem,
        icon: const Icon(Icons.add),
        label: const Text('Item'),
      ),
      body: FutureBuilder<List<Map<String, Object?>>>(
        future: AppDatabase.instance.getInventoryItems(),
        builder: (context, snapshot) {
          if (!snapshot.hasData) {
            return const Center(child: CircularProgressIndicator());
          }

          final items = snapshot.data!;

          if (items.isEmpty) {
            return const Center(
              child: Text('Add appliances, furniture or other home items.'),
            );
          }

          return ListView.builder(
            padding: const EdgeInsets.fromLTRB(12, 12, 12, 100),
            itemCount: items.length,
            itemBuilder: (context, index) {
              final row = items[index];
              final warranty = row['warranty_expiry'] as String?;
              final price = row['purchase_price'];

              return Card(
                child: ExpansionTile(
                  leading: const CircleAvatar(
                    child: Icon(Icons.inventory_2_outlined),
                  ),
                  title: Text(row['name'] as String),
                  subtitle: Text(
                    '${row['room_name'] ?? 'No room'}'
                    '${(row['category'] as String).isEmpty ? '' : ' • ${row['category']}'}',
                  ),
                  childrenPadding:
                      const EdgeInsets.fromLTRB(16, 0, 16, 14),
                  children: [
                    if (price != null)
                      ListTile(
                        contentPadding: EdgeInsets.zero,
                        title: const Text('Purchase price'),
                        trailing: Text(money.format((price as num).toDouble())),
                      ),
                    if (row['purchase_date'] != null)
                      ListTile(
                        contentPadding: EdgeInsets.zero,
                        title: const Text('Purchased'),
                        trailing: Text(
                          DateFormat('d MMM yyyy').format(
                            DateTime.parse(row['purchase_date'] as String),
                          ),
                        ),
                      ),
                    if (warranty != null)
                      ListTile(
                        contentPadding: EdgeInsets.zero,
                        title: const Text('Warranty expires'),
                        trailing: Text(
                          DateFormat('d MMM yyyy')
                              .format(DateTime.parse(warranty)),
                        ),
                      ),
                    if ((row['serial_number'] as String).isNotEmpty)
                      ListTile(
                        contentPadding: EdgeInsets.zero,
                        title: const Text('Serial number'),
                        trailing: SizedBox(
                          width: 150,
                          child: Text(
                            row['serial_number'] as String,
                            textAlign: TextAlign.end,
                            overflow: TextOverflow.ellipsis,
                          ),
                        ),
                      ),
                    if ((row['notes'] as String).isNotEmpty)
                      ListTile(
                        contentPadding: EdgeInsets.zero,
                        title: const Text('Notes'),
                        subtitle: Text(row['notes'] as String),
                      ),
                    Align(
                      alignment: Alignment.centerRight,
                      child: TextButton.icon(
                        onPressed: () async {
                          await AppDatabase.instance
                              .deleteInventoryItem(row['id'] as int);
                          if (mounted) setState(() {});
                        },
                        icon: const Icon(Icons.delete_outline),
                        label: const Text('Delete'),
                      ),
                    ),
                  ],
                ),
              );
            },
          );
        },
      ),
    );
  }
}


Writing /content/home_organiser/lib/screens/inventory_screen.dart


In [14]:
# @title
%%writefile /content/home_organiser/lib/services/settings_controller.dart
import 'package:flutter/material.dart';

import '../db/app_database.dart';

class SettingsController extends ChangeNotifier {
  SettingsController._();

  static final SettingsController instance = SettingsController._();

  ThemeMode _themeMode = ThemeMode.system;
  ThemeMode get themeMode => _themeMode;

  Future<void> load() async {
    final value = await AppDatabase.instance.getSetting(
      'theme_mode',
      fallback: 'system',
    );
    _themeMode = _fromString(value);
  }

  Future<void> setThemeMode(ThemeMode mode) async {
    _themeMode = mode;
    await AppDatabase.instance.setSetting('theme_mode', _toString(mode));
    notifyListeners();
  }

  static ThemeMode _fromString(String value) {
    switch (value) {
      case 'light':
        return ThemeMode.light;
      case 'dark':
        return ThemeMode.dark;
      default:
        return ThemeMode.system;
    }
  }

  static String _toString(ThemeMode mode) {
    switch (mode) {
      case ThemeMode.light:
        return 'light';
      case ThemeMode.dark:
        return 'dark';
      case ThemeMode.system:
        return 'system';
    }
  }
}


Writing /content/home_organiser/lib/services/settings_controller.dart


In [15]:
# @title
%%writefile /content/home_organiser/lib/services/notification_service.dart
import 'dart:ui';

import 'package:flutter/widgets.dart';
import 'package:flutter_local_notifications/flutter_local_notifications.dart';
import 'package:workmanager/workmanager.dart';

import '../db/app_database.dart';

const _dailyWorkName = 'my_home_daily_reminder_work';
const _dailyTaskName = 'my_home_daily_reminder_task';
const _channelId = 'my_home_daily';
const _channelName = 'Daily My Home reminders';
const _channelDescription = 'Daily reminders for chores, project tasks and bills';

@pragma('vm:entry-point')
void notificationCallbackDispatcher() {
  Workmanager().executeTask((task, inputData) async {
    WidgetsFlutterBinding.ensureInitialized();
    DartPluginRegistrant.ensureInitialized();
    try {
      await NotificationService.initializeLocalNotifications();
      if (task == _dailyTaskName) {
        await NotificationService.runDailyCheck();
      }
      return true;
    } catch (_) {
      return false;
    }
  });
}

class NotificationService {
  static final FlutterLocalNotificationsPlugin _notifications =
      FlutterLocalNotificationsPlugin();

  static bool _localInitialized = false;
  static bool _workmanagerInitialized = false;

  static const NotificationDetails _details = NotificationDetails(
    android: AndroidNotificationDetails(
      _channelId,
      _channelName,
      channelDescription: _channelDescription,
      importance: Importance.high,
      priority: Priority.high,
    ),
  );

  static Future<void> initialize() async {
    try {
      await initializeLocalNotifications();
    } catch (_) {
      // Notifications must never prevent the rest of the app from opening.
    }
  }

  static Future<void> initializeLocalNotifications() async {
    if (_localInitialized) {
      return;
    }

    // @mipmap/ic_launcher is generated by `flutter create`, so unlike a custom
    // runtime-only drawable it is guaranteed to exist. Cell 2B also keeps it
    // from being removed by release resource shrinking.
    const initializationSettings = InitializationSettings(
      android: AndroidInitializationSettings('@mipmap/ic_launcher'),
    );

    final initialized = await _notifications.initialize(
      settings: initializationSettings,
    );
    if (initialized == false) {
      throw StateError('Local notification plugin failed to initialise.');
    }

    _localInitialized = true;
  }

  static Future<AndroidFlutterLocalNotificationsPlugin?> _androidPlugin() async {
    await initializeLocalNotifications();
    return _notifications.resolvePlatformSpecificImplementation<
        AndroidFlutterLocalNotificationsPlugin>();
  }

  static Future<bool> areNotificationsEnabled() async {
    try {
      final android = await _androidPlugin();
      if (android == null) {
        return false;
      }
      return await android.areNotificationsEnabled() ?? false;
    } catch (_) {
      return false;
    }
  }

  static Future<bool> requestPermission() async {
    try {
      final android = await _androidPlugin();
      if (android == null) {
        return false;
      }

      // Android 13+ displays the POST_NOTIFICATIONS runtime prompt here.
      // On older Android versions this simply reports the current state.
      final granted = await android.requestNotificationsPermission();
      if (granted != null) {
        return granted;
      }
      return await android.areNotificationsEnabled() ?? false;
    } catch (_) {
      return false;
    }
  }

  static Future<void> _ensureWorkmanagerInitialized() async {
    if (_workmanagerInitialized) {
      return;
    }
    await Workmanager().initialize(notificationCallbackDispatcher);
    _workmanagerInitialized = true;
  }

  static Future<String?> rescheduleDailyCheck() async {
    try {
      final settings = await AppDatabase.instance.getSettings();
      if (settings['notifications_enabled'] != '1') {
        if (_workmanagerInitialized) {
          try {
            await Workmanager().cancelByUniqueName(_dailyWorkName);
          } catch (_) {}
        }
        return null;
      }

      if (!await areNotificationsEnabled()) {
        return 'Android notification permission is not enabled.';
      }

      await _ensureWorkmanagerInitialized();
      await Workmanager().cancelByUniqueName(_dailyWorkName);

      final hour = int.tryParse(settings['notification_hour'] ?? '') ?? 9;
      final minute = int.tryParse(settings['notification_minute'] ?? '') ?? 0;
      final now = DateTime.now();
      var first = DateTime(now.year, now.month, now.day, hour, minute);
      if (!first.isAfter(now)) {
        first = first.add(const Duration(days: 1));
      }

      await Workmanager().registerPeriodicTask(
        _dailyWorkName,
        _dailyTaskName,
        frequency: const Duration(hours: 24),
        initialDelay: first.difference(now),
      );
      return null;
    } catch (e) {
      return 'Daily reminder scheduling failed: $e';
    }
  }

  static Future<void> runDailyCheck() async {
    final settings = await AppDatabase.instance.getSettings();
    if (settings['notifications_enabled'] != '1') {
      return;
    }

    await initializeLocalNotifications();
    final counts = await AppDatabase.instance.getReminderCounts();
    final parts = <String>[];

    void addPart(
      String key,
      String settingKey,
      String singular,
      String plural,
    ) {
      if (settings[settingKey] != '1') {
        return;
      }
      final count = counts[key] ?? 0;
      if (count > 0) {
        parts.add('$count ${count == 1 ? singular : plural}');
      }
    }

    addPart('overdue', 'notify_overdue', 'overdue item', 'overdue items');
    addPart(
      'due_today',
      'notify_due_today',
      'item due today',
      'items due today',
    );
    addPart(
      'pending_projects',
      'notify_pending_projects',
      'pending project task',
      'pending project tasks',
    );
    addPart(
      'upcoming_bills',
      'notify_upcoming_bills',
      'bill due soon',
      'bills due soon',
    );

    if (parts.isEmpty) {
      return;
    }

    await _notifications.show(
      id: 4501,
      title: 'My Home reminder',
      body: parts.join(' • '),
      notificationDetails: _details,
    );
  }

  static Future<void> showTestNotification() async {
    await initializeLocalNotifications();

    if (!await areNotificationsEnabled()) {
      throw StateError(
        'Android notification permission is not enabled for My Home.',
      );
    }

    await _notifications.show(
      id: 4502,
      title: 'My Home notifications are ready',
      body: 'Your daily reminder notifications are working.',
      notificationDetails: _details,
    );
  }
}


Writing /content/home_organiser/lib/services/notification_service.dart


In [16]:
# @title
%%writefile /content/home_organiser/lib/screens/settings_screen.dart
import 'package:flutter/material.dart';

import '../db/app_database.dart';
import '../services/notification_service.dart';
import '../services/settings_controller.dart';

class SettingsScreen extends StatefulWidget {
  const SettingsScreen({super.key});

  @override
  State<SettingsScreen> createState() => _SettingsScreenState();
}

class _SettingsScreenState extends State<SettingsScreen> {
  Map<String, String>? _settings;
  bool _saving = false;

  @override
  void initState() {
    super.initState();
    _load();
  }

  Future<void> _load() async {
    final settings = await AppDatabase.instance.getSettings();
    if (mounted) {
      setState(() => _settings = settings);
    }
  }

  bool _on(String key) => _settings?[key] == '1';

  Future<void> _setToggle(String key, bool value) async {
    if (_settings == null) {
      return;
    }
    setState(() { _settings![key] = value ? '1' : '0'; _saving = true; });
    await AppDatabase.instance.setSetting(key, value ? '1' : '0');
    final error = await NotificationService.rescheduleDailyCheck();
    if (!mounted) {
      return;
    }
    setState(() => _saving = false);
    if (error != null) {
      ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(error)));
    }
  }

  Future<void> _setNotificationsEnabled(bool value) async {
    if (_settings == null) {
      return;
    }
    setState(() { _settings!['notifications_enabled'] = value ? '1' : '0'; _saving = true; });
    await AppDatabase.instance.setSetting('notifications_enabled', value ? '1' : '0');
    if (!value) {
      await NotificationService.rescheduleDailyCheck();
      if (mounted) {
        setState(() => _saving = false);
      }
      return;
    }
    final granted = await NotificationService.requestPermission();
    if (!granted) {
      await AppDatabase.instance.setSetting('notifications_enabled', '0');
      if (!mounted) {
      return;
    }
      setState(() { _settings!['notifications_enabled'] = '0'; _saving = false; });
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Notification permission is off. Allow notifications for My Home and try again.')));
      return;
    }
    final error = await NotificationService.rescheduleDailyCheck();
    if (!mounted) {
      return;
    }
    setState(() => _saving = false);
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(error ?? 'Daily notifications enabled successfully.')));
  }

  Future<void> _pickTime() async {
    if (_settings == null) {
      return;
    }
    final initial = TimeOfDay(
      hour: int.tryParse(_settings!['notification_hour'] ?? '') ?? 9,
      minute: int.tryParse(_settings!['notification_minute'] ?? '') ?? 0,
    );
    final picked = await showTimePicker(context: context, initialTime: initial);
    if (picked == null) {
      return;
    }
    setState(() {
      _settings!['notification_hour'] = picked.hour.toString();
      _settings!['notification_minute'] = picked.minute.toString();
      _saving = true;
    });
    await AppDatabase.instance.setSetting('notification_hour', picked.hour.toString());
    await AppDatabase.instance.setSetting('notification_minute', picked.minute.toString());
    await NotificationService.rescheduleDailyCheck();
    if (mounted) {
      setState(() => _saving = false);
    }
  }

  String _timeText(BuildContext context) {
    final hour = int.tryParse(_settings?['notification_hour'] ?? '') ?? 9;
    final minute = int.tryParse(_settings?['notification_minute'] ?? '') ?? 0;
    return TimeOfDay(hour: hour, minute: minute).format(context);
  }

  @override
  Widget build(BuildContext context) {
    final settings = _settings;
    if (settings == null) {
      return const Scaffold(body: Center(child: CircularProgressIndicator()));
    }

    return Scaffold(
      appBar: AppBar(title: const Text('Settings')),
      body: ListView(
        padding: const EdgeInsets.fromLTRB(16, 16, 16, 40),
        children: [
          Text('Appearance', style: Theme.of(context).textTheme.titleLarge),
          const SizedBox(height: 8),
          Card(
            child: Padding(
              padding: const EdgeInsets.all(12),
              child: SegmentedButton<ThemeMode>(
                segments: const [
                  ButtonSegment(value: ThemeMode.system, label: Text('System'), icon: Icon(Icons.settings_brightness)),
                  ButtonSegment(value: ThemeMode.light, label: Text('Light'), icon: Icon(Icons.light_mode_outlined)),
                  ButtonSegment(value: ThemeMode.dark, label: Text('Dark'), icon: Icon(Icons.dark_mode_outlined)),
                ],
                selected: {SettingsController.instance.themeMode},
                onSelectionChanged: (selection) async {
                  await SettingsController.instance.setThemeMode(selection.first);
                  if (mounted) {
                    setState(() {});
                  }
                },
              ),
            ),
          ),
          const SizedBox(height: 22),
          Text('Daily reminders', style: Theme.of(context).textTheme.titleLarge),
          const SizedBox(height: 8),
          Card(
            child: Column(
              children: [
                SwitchListTile(
                  value: _on('notifications_enabled'),
                  title: const Text('Enable daily notifications'),
                  subtitle: const Text('Checks your selected categories once each day.'),
                  onChanged: _saving ? null : _setNotificationsEnabled,
                ),
                const Divider(height: 1),
                ListTile(
                  enabled: _on('notifications_enabled') && !_saving,
                  leading: const Icon(Icons.schedule),
                  title: const Text('Reminder time'),
                  subtitle: Text('${_timeText(context)} • Android may deliver background work a little later to save battery.'),
                  trailing: const Icon(Icons.chevron_right),
                  onTap: _on('notifications_enabled') && !_saving ? _pickTime : null,
                ),
              ],
            ),
          ),
          const SizedBox(height: 12),
          Text('Notify me about', style: Theme.of(context).textTheme.titleMedium),
          Card(
            child: Column(
              children: [
                SwitchListTile(
                  value: _on('notify_overdue'),
                  title: const Text('Overdue'),
                  subtitle: const Text('Overdue chores and project tasks'),
                  onChanged: _on('notifications_enabled') && !_saving
                      ? (value) => _setToggle('notify_overdue', value)
                      : null,
                ),
                const Divider(height: 1),
                SwitchListTile(
                  value: _on('notify_due_today'),
                  title: const Text('Due today'),
                  subtitle: const Text('Chores and project tasks due today'),
                  onChanged: _on('notifications_enabled') && !_saving
                      ? (value) => _setToggle('notify_due_today', value)
                      : null,
                ),
                const Divider(height: 1),
                SwitchListTile(
                  value: _on('notify_pending_projects'),
                  title: const Text('Pending project tasks'),
                  subtitle: const Text('Open tasks that are not yet due or have no due date'),
                  onChanged: _on('notifications_enabled') && !_saving
                      ? (value) => _setToggle('notify_pending_projects', value)
                      : null,
                ),
                const Divider(height: 1),
                SwitchListTile(
                  value: _on('notify_upcoming_bills'),
                  title: const Text('Upcoming bills'),
                  subtitle: const Text('Bills due within the next 3 days'),
                  onChanged: _on('notifications_enabled') && !_saving
                      ? (value) => _setToggle('notify_upcoming_bills', value)
                      : null,
                ),
              ],
            ),
          ),
          const SizedBox(height: 12),
          OutlinedButton.icon(
            onPressed: _on('notifications_enabled') && !_saving
                ? () async {
                    final messenger = ScaffoldMessenger.of(context);
                    try {
                      await NotificationService.showTestNotification();
                      if (!mounted) {
      return;
    }
                      messenger.showSnackBar(const SnackBar(content: Text('Test notification sent.')));
                    } catch (e) {
                      if (!mounted) {
      return;
    }
                      messenger.showSnackBar(SnackBar(content: Text('Test notification failed: $e')));
                    }
                  }
                : null,
            icon: const Icon(Icons.notifications_active_outlined),
            label: const Text('Send test notification'),
          ),
          if (_saving) ...[
            const SizedBox(height: 12),
            const LinearProgressIndicator(),
          ],
        ],
      ),
    );
  }
}


Writing /content/home_organiser/lib/screens/settings_screen.dart


In [17]:
# @title

%%writefile /content/home_organiser/lib/screens/more_screen.dart
import 'package:flutter/material.dart';

import '../services/backup_service.dart';
import 'bills_screen.dart';
import 'inventory_screen.dart';
import 'settings_screen.dart';

class MoreScreen extends StatefulWidget {
  const MoreScreen({super.key});

  @override
  State<MoreScreen> createState() => _MoreScreenState();
}

class _MoreScreenState extends State<MoreScreen> {
  bool _busy = false;

  Future<void> _backup() async {
    setState(() => _busy = true);
    try {
      final saved = await BackupService.saveBackup();
      if (!mounted) return;
      ScaffoldMessenger.of(context).showSnackBar(
        SnackBar(
          content: Text(saved ? 'Backup saved.' : 'Backup cancelled.'),
        ),
      );
    } catch (e) {
      if (!mounted) return;
      ScaffoldMessenger.of(context).showSnackBar(
        SnackBar(content: Text('Backup failed: $e')),
      );
    } finally {
      if (mounted) setState(() => _busy = false);
    }
  }

  Future<void> _restore() async {
    final confirmed = await showDialog<bool>(
      context: context,
      builder: (context) => AlertDialog(
        title: const Text('Restore backup?'),
        content: const Text(
          'This replaces the current local data with the contents of the '
          'selected backup file. Create a new backup first if you want to '
          'keep the current data.',
        ),
        actions: [
          TextButton(
            onPressed: () => Navigator.pop(context, false),
            child: const Text('Cancel'),
          ),
          FilledButton(
            onPressed: () => Navigator.pop(context, true),
            child: const Text('Choose backup'),
          ),
        ],
      ),
    );

    if (confirmed != true) return;

    setState(() => _busy = true);
    try {
      final restored = await BackupService.restoreBackup();
      if (!mounted) return;
      ScaffoldMessenger.of(context).showSnackBar(
        SnackBar(
          content: Text(
            restored
                ? 'Backup restored. Older backup formats are upgraded automatically.'
                : 'Restore cancelled.',
          ),
        ),
      );
    } catch (e) {
      if (!mounted) return;
      ScaffoldMessenger.of(context).showSnackBar(
        SnackBar(content: Text('Restore failed: $e')),
      );
    } finally {
      if (mounted) setState(() => _busy = false);
    }
  }

  void _open(Widget page) {
    Navigator.push(context, MaterialPageRoute(builder: (_) => page));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('More')),
      body: ListView(
        padding: const EdgeInsets.all(16),
        children: [
          _MoreCard(
            icon: Icons.receipt_long_outlined,
            title: 'Bills',
            subtitle: 'Recurring household bills and due dates',
            onTap: () => _open(const BillsScreen()),
          ),
          _MoreCard(
            icon: Icons.inventory_2_outlined,
            title: 'Inventory',
            subtitle: 'Appliances, warranties, serial numbers and purchases',
            onTap: () => _open(const InventoryScreen()),
          ),
          _MoreCard(
            icon: Icons.settings_outlined,
            title: 'Settings',
            subtitle: 'Dark mode, daily reminders and notification categories',
            onTap: () => _open(const SettingsScreen()),
          ),
          const SizedBox(height: 18),
          Text('Backup & restore',
              style: Theme.of(context).textTheme.titleLarge),
          const SizedBox(height: 8),
          Card(
            child: Column(
              children: [
                ListTile(
                  enabled: !_busy,
                  leading: const Icon(Icons.backup_outlined),
                  title: const Text('Save JSON backup'),
                  subtitle: const Text(
                    'Choose where to save a complete copy of your app data.',
                  ),
                  onTap: _busy ? null : _backup,
                ),
                const Divider(height: 1),
                ListTile(
                  enabled: !_busy,
                  leading: const Icon(Icons.restore_outlined),
                  title: const Text('Restore JSON backup'),
                  subtitle: const Text(
                    'Replace the current database with a saved backup.',
                  ),
                  onTap: _busy ? null : _restore,
                ),
              ],
            ),
          ),
          if (_busy) ...[
            const SizedBox(height: 12),
            const LinearProgressIndicator(),
          ],
          const SizedBox(height: 22),
          const Card(
            child: ListTile(
              leading: Icon(Icons.lock_outline),
              title: Text('Local-only app'),
              subtitle: Text(
                'No account, cloud database, advertising, analytics or home APIs.',
              ),
            ),
          ),
        ],
      ),
    );
  }
}

class _MoreCard extends StatelessWidget {
  final IconData icon;
  final String title;
  final String subtitle;
  final VoidCallback onTap;

  const _MoreCard({
    required this.icon,
    required this.title,
    required this.subtitle,
    required this.onTap,
  });

  @override
  Widget build(BuildContext context) {
    return Card(
      child: ListTile(
        leading: CircleAvatar(child: Icon(icon)),
        title: Text(title),
        subtitle: Text(subtitle),
        trailing: const Icon(Icons.chevron_right),
        onTap: onTap,
      ),
    );
  }
}


Writing /content/home_organiser/lib/screens/more_screen.dart


In [18]:
# @title
%%writefile /content/home_organiser/lib/main.dart
import 'package:flutter/material.dart';

import 'services/notification_service.dart';
import 'services/settings_controller.dart';
import 'screens/budget_screen.dart';
import 'screens/dashboard_screen.dart';
import 'screens/more_screen.dart';
import 'screens/projects_screen.dart';
import 'screens/rooms_screen.dart';

Future<void> main() async {
  WidgetsFlutterBinding.ensureInitialized();

  // Load saved theme if possible, but never let startup data/plugin errors
  // stop the application from rendering.
  try {
    await SettingsController.instance.load();
  } catch (_) {}

  runApp(const MyHomeApp());

  // Notification initialisation is deliberately non-blocking.
  // The app must remain usable even if a device rejects notification setup.
  NotificationService.initialize();
}

class MyHomeApp extends StatelessWidget {
  const MyHomeApp({super.key});

  ThemeData _theme(Brightness brightness) {
    return ThemeData(
      colorScheme: ColorScheme.fromSeed(
        seedColor: const Color(0xFF315A73),
        brightness: brightness,
      ),
      useMaterial3: true,
      inputDecorationTheme: const InputDecorationTheme(
        border: OutlineInputBorder(),
      ),
    );
  }

  @override
  Widget build(BuildContext context) {
    return AnimatedBuilder(
      animation: SettingsController.instance,
      builder: (context, _) => MaterialApp(
        debugShowCheckedModeBanner: false,
        title: 'My Home',
        theme: _theme(Brightness.light),
        darkTheme: _theme(Brightness.dark),
        themeMode: SettingsController.instance.themeMode,
        home: const HomeShell(),
      ),
    );
  }
}

class HomeShell extends StatefulWidget {
  const HomeShell({super.key});

  @override
  State<HomeShell> createState() => _HomeShellState();
}

class _HomeShellState extends State<HomeShell> {
  int _selectedIndex = 0;
  int _navigationVersion = 0;

  List<Widget> _buildPages() => [
        DashboardScreen(key: ValueKey('dashboard-$_navigationVersion')),
        RoomsScreen(key: ValueKey('rooms-$_navigationVersion')),
        BudgetScreen(key: ValueKey('budget-$_navigationVersion')),
        ProjectsScreen(key: ValueKey('projects-$_navigationVersion')),
        MoreScreen(key: ValueKey('more-$_navigationVersion')),
      ];

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      body: IndexedStack(
        index: _selectedIndex,
        children: _buildPages(),
      ),
      bottomNavigationBar: NavigationBar(
        selectedIndex: _selectedIndex,
        onDestinationSelected: (index) {
          setState(() {
            _selectedIndex = index;
            _navigationVersion++;
          });
        },
        destinations: const [
          NavigationDestination(
            icon: Icon(Icons.dashboard_outlined),
            selectedIcon: Icon(Icons.dashboard),
            label: 'Home',
          ),
          NavigationDestination(
            icon: Icon(Icons.meeting_room_outlined),
            selectedIcon: Icon(Icons.meeting_room),
            label: 'Rooms',
          ),
          NavigationDestination(
            icon: Icon(Icons.account_balance_wallet_outlined),
            selectedIcon: Icon(Icons.account_balance_wallet),
            label: 'Money',
          ),
          NavigationDestination(
            icon: Icon(Icons.construction_outlined),
            selectedIcon: Icon(Icons.construction),
            label: 'Projects',
          ),
          NavigationDestination(
            icon: Icon(Icons.more_horiz),
            selectedIcon: Icon(Icons.more),
            label: 'More',
          ),
        ],
      ),
    );
  }
}


Overwriting /content/home_organiser/lib/main.dart


## Preflight check

Run the next cell before building. The desired result is:

`No issues found!`

Messages saying newer package versions exist are informational and can be ignored.

In [19]:
# CELL 13 — Verify notification resources, clean, format and analyse

from pathlib import Path

PROJECT = "/content/home_organiser"
project_path = Path(PROJECT)
res_path = project_path / "android/app/src/main/res"
manifest_path = project_path / "android/app/src/main/AndroidManifest.xml"

# Fail here, before the APK build, if any Android notification prerequisite is missing.
launcher_icons = list(res_path.glob("mipmap-*/ic_launcher.png")) + list(
    res_path.glob("mipmap-*/ic_launcher.webp")
)
assert launcher_icons, "Missing Flutter launcher icon resource (ic_launcher)."
assert (res_path / "raw/keep.xml").exists(), "Missing res/raw/keep.xml resource keep rules."
assert (res_path / "drawable/my_home_icon.xml").exists(), "Missing custom My Home launcher icon."
assert "android.permission.POST_NOTIFICATIONS" in manifest_path.read_text(), (
    "POST_NOTIFICATIONS permission is missing from AndroidManifest.xml."
)

print("Notification Android resources verified.")

# Rebuild Flutter/Dart package metadata from scratch.
!cd {PROJECT} && flutter clean
!rm -rf {PROJECT}/.dart_tool
!rm -f {PROJECT}/pubspec.lock
!cd {PROJECT} && flutter pub get

# Show the exact notification package versions used by this build.
!cd {PROJECT} && flutter pub deps --style=compact | grep -E "flutter_local_notifications|workmanager" || true

# Format only after package metadata has been rebuilt.
!cd {PROJECT} && dart format lib

# Analyse the complete Flutter project.
!cd {PROJECT} && flutter analyze

print("Continue only if Flutter reports 'No issues found!'.")


Notification Android resources verified.
Deleting .dart_tool...                                                 ⣷      3ms
Deleting .flutter-plugins-dependencies...                              ⣷      0ms
Resolving dependencies...
+ android_file_picker 2.1.2
+ args 2.7.0
+ async 2.13.1
+ boolean_selector 2.1.2
+ characters 1.4.1
+ clock 1.1.3
+ collection 1.19.1
+ cross_file 0.3.5+5 (0.4.0 available)
+ cupertino_icons 1.0.9 (2.0.0 available)
+ dbus 0.8.0
+ fake_async 1.3.3
+ ffi 2.2.0
+ ffi_leak_tracker 0.1.2
+ file_picker 13.3.0
+ file_picker_darwin 2.4.0
+ file_picker_linux 2.0.2
+ file_picker_platform_interface 4.2.0
+ file_picker_web 4.2.0
+ flutter 0.0.0 from sdk flutter
+ flutter_lints 6.0.0
+ flutter_local_notifications 22.3.3
+ flutter_local_notifications_linux 8.0.2
+ flutter_local_notifications_platform_interface 12.2.0
+ flutter_local_notifications_web 1.0.0
+ flutter_local_notifications_windows 3.1.1
+ flutter_test 0.0.0 from sdk flutter
+ flutter_web_plug

In [20]:
# @title

# CELL 14 — Build release APK

PROJECT = "/content/home_organiser"

!cd {PROJECT} && flutter build apk --release

!cp {PROJECT}/build/app/outputs/flutter-apk/app-release.apk /content/MyHome_v4_5_7.apk
!ls -lh /content/MyHome_v4_5_7.apk

print("APK created: /content/MyHome_v4_5_7.apk")


[1/1] Android SDK
  ├─ [1/6] android-arm-profile/linux-x64
  ├─ [1/6] android-arm-profile/linux-x64                         (2.5MB in 0.5s)
  ├─ [2/6] android-arm-release/linux-x64
  ├─ [2/6] android-arm-release/linux-x64                         (2.0MB in 0.2s)
  ├─ [3/6] android-arm64-profile/linux-x64
  ├─ [3/6] android-arm64-profile/linux-x64                       (2.5MB in 0.2s)
  ├─ [4/6] android-arm64-release/linux-x64
  ├─ [4/6] android-arm64-release/linux-x64                       (2.1MB in 0.2s)
  ├─ [5/6] android-x64-profile/linux-x64
  ├─ [5/6] android-x64-profile/linux-x64                         (2.5MB in 0.2s)
  └─ [6/6] android-x64-release/linux-x64
  └─ [6/6] android-x64-release/linux-x64                         (2.1MB in 0.1s)
The 'android' binary can also be found in the cmdline-tools directory, and 'android sdk' is the replacement for 'sdkmanager'.
To learn more about the Android CLI and how to use it, see the documentation (https://d.android.com/tools/agents/android


## Important: keep the Android signing key

Flutter's generated Android project signs this personal build with the local debug keystore.

Android will only install a future version **over** the current version if it is signed by the same key. Colab runtimes are temporary, so download a copy of the signing key after a successful build and keep it somewhere safe.

If you are building v3 in the **same Colab runtime** that built your previous APK, do not replace the existing `/root/.android/debug.keystore` before building.

On a future fresh Colab runtime, upload your saved key and copy it back to:

`/root/.android/debug.keystore`

before running the APK build cell.


In [ ]:

# OPTIONAL BUT STRONGLY RECOMMENDED — save the Android signing key
# Keep this file private. It lets future APK versions install over this one.

from google.colab import files
from pathlib import Path
import shutil

source_key = Path.home() / ".android" / "debug.keystore"
saved_key = Path("/content/MyHome_signing_key.keystore")

if source_key.exists():
    shutil.copy2(source_key, saved_key)
    print("Signing key copied to:", saved_key)
    files.download(str(saved_key))
else:
    print("No debug keystore exists yet. Run the APK build cell first, then run this cell again.")



### Using the saved signing key in a future Colab session

Only use the following optional cell on a **future/new runtime** when you already have `MyHome_signing_key.keystore`. Run it **before** building the APK.


In [ ]:

# OPTIONAL FUTURE-RUNTIME CELL — restore your saved signing key
# Do NOT run this unless you are deliberately uploading the key saved from an earlier build.

# from google.colab import files
# from pathlib import Path
# import shutil
#
# uploaded = files.upload()
# uploaded_name = next(iter(uploaded))
# target = Path.home() / ".android" / "debug.keystore"
# target.parent.mkdir(parents=True, exist_ok=True)
# shutil.copy2(uploaded_name, target)
# target.chmod(0o600)
# print("Signing key restored:", target)


In [21]:

# FINAL CELL — Download APK

from google.colab import files
files.download("/content/MyHome_v4_5_7.apk")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## Updating the existing app

The package name remains `uk.co.local.home_organiser`, so Android can install v4.5 over the earlier app when it is signed with the same key.

The SQLite database is upgraded to **version 4**. Existing rooms, chores, budgets, expenses, bill categories, bills, projects, project tasks, project expenses and inventory are retained. Existing projects default to **Home improvement** and existing project tasks simply have no due date until you add one to new tasks.

## Backup compatibility

v4.5 exports schema version 4. Restore remains intentionally backward-compatible:

- v4.2/v4.3/v4.4 backups can still be restored.
- Missing v4.5 columns use database defaults.
- Missing settings are recreated with safe defaults.
- Unknown columns are ignored.
- Existing IDs and relationships are preserved.
- Old bill-category migration from v4.4 is retained.

## Recommended v4.5 test

1. Install v4.5 over v4.4 and confirm your existing data remains.
2. Confirm the launcher now uses the custom **My Home house/check icon**.
3. Open **More → Settings** and switch between System, Light and Dark.
4. Enable notifications and allow Android notification permission.
5. Choose the reminder categories you want and use **Send test notification**.
6. Add a **Learning / skill** project, for example “Learn Python”.
7. Add project tasks both with and without due dates, including a test due date of today.
8. Restore an older v4.2/v4.3/v4.4 JSON backup and confirm it imports without errors.

### Notification timing note

The daily reminder uses Android WorkManager. The selected time is the requested start time, but Android may defer background work slightly for battery optimisation. This is expected behaviour rather than an app error.
